# Scraping and Data Source Ideas
This notebook collects source ideas for dry-bulk shipping and related data, plus a reusable scraped-source design pattern.

## Initialising packages

In [12]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import tqdm
import pandas as pd
import re
import time
from datetime import datetime


## Break wave

In [13]:
def extract_links(url):
    try:
        # Send a GET request to the webpage
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for HTTP errors
        
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find all anchor tags and extract href attributes
        links = []
        for anchor in soup.find_all('a', href=True):
            href = anchor['href']
            # Convert relative URLs to absolute URLs
            absolute_url = urljoin(url, href)
            links.append(absolute_url)
        
        # next_pg = soup.find("a", rel="next")
        return links
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the webpage: {e}")
        return []


In [14]:
breakwave_links_article = []
# URL of the webpage to scrape
target_url = "https://www.breakwaveadvisors.com/insights"

# for i in tqdm.tqdm(range(0,1000)): # I dnont want to extract all everytime
for i in tqdm.tqdm(range(0,50)):
    try:
        # Extract links from the webpage
        extracted_links = list(set(extract_links(target_url)))

        #only extract link containing "/2025/" or "/2024/" or "/2023/"
        pattern = r'/202[0-5]'
        breakwave_links_article = breakwave_links_article + [link for link in extracted_links if re.search(pattern, link)]
        # move to the next page
        extracted_links_nxt_pg = [link for link in extracted_links if "?offset=" in link and 'reversePaginate' not in link and '#header' not in link]
        target_url = extracted_links_nxt_pg[0]
        if target_url is None:
            break
    except Exception as e:
        print(f"Error processing page {i}: {e}")
        break

100%|██████████| 50/50 [01:31<00:00,  1.82s/it]


In [15]:
print(f"Found {len(breakwave_links_article)}:")
for i, link in enumerate(breakwave_links_article, 1):
    print(f"{i}. {link}")

Found 306:
1. https://www.breakwaveadvisors.com/insights/2024/10/25/breakwave-weekend-read-547kk-rw7zy-awmfb-mf26k-dk29d-56ysb-x3jbe-xbwk9-dnpbr-b6jhc-l4mb2-d4l2e-f7l54-fsr59-khctl
2. https://www.breakwaveadvisors.com/insights/2024/10/25/breakwave-weekend-read-547kk-rw7zy-awmfb-mf26k-dk29d-56ysb-x3jbe-xbwk9-dnpbr-b6jhc-l4mb2-d4l2e-f7l54-fsr59-y9ckz
3. https://www.breakwaveadvisors.com/insights/2024/10/25/breakwave-weekend-read-547kk-rw7zy-awmfb-mf26k-dk29d-56ysb-x3jbe-xbwk9-dnpbr-b6jhc-l4mb2-d4l2e-f7l54-fsr59
4. https://www.breakwaveadvisors.com/insights/2024/10/25/breakwave-weekend-read-547kk-rw7zy-awmfb-mf26k-dk29d-56ysb-x3jbe-xbwk9-dnpbr-b6jhc-l4mb2-d4l2e-f7l54
5. https://www.breakwaveadvisors.com/insights/2025/12/23/demand-recap-2025-3cs-countries-commodities-capacity
6. https://www.breakwaveadvisors.com/insights/2025/12/24/bauxite-helps-to-hold-the-capesize-market
7. https://www.breakwaveadvisors.com/insights/2025/12/23/dry-bulk-market-on-solid-footing
8. https://www.breakwaveadvi

### Content from Extracting article

In [ ]:
import requests
from bs4 import BeautifulSoup

def extract_article_text(url):
    try:
        # Send a GET request to the webpage
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise an exception for HTTP errors
        
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract the main article content
        article_title = soup.find('div', class_='post-title').get_text(strip=True) if soup.find('div', class_='post-title') else "No title found"
        article_date = soup.find('div', class_='entry-dateline').get_text(strip=True) if soup.find('div', class_='entry-dateline') else "No date found"

        # Find ALL divs with class 'sqs-html-content' (not just the first one)
        article_bodies = soup.find_all('div', class_='sqs-html-content')
        
        # Combine text from all matching divs
        all_text_parts = []
        for body in article_bodies:
            # Get text from each div and add to our list
            text_content = ' '.join(body.stripped_strings)
            if text_content:  # Only add if there's actual text
                all_text_parts.append(text_content)
        
        # Join all text parts with spaces
        article_text = ' '.join(all_text_parts)
        
        if not article_text:
            article_text = "Could not find main article content"
        
        return {
            'title': article_title,
            'date': article_date,
            'text': article_text,
            'bodies_found': len(article_bodies),  # Number of divs found
            'url': url
        }
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the webpage: {e}")
        return None


#===== testing ====================
# URL of the article to scrape
article_url = "https://www.breakwaveadvisors.com/insights/2025/7/24/spotlight-on-brazilian-iron-ore-shipments-to-the-far-east"

# Extract text from the article
article_data = extract_article_text(article_url)

if article_data:
    print("Title:", article_data['title'])
    print("Date:", article_data['date'])
    print(f"Found {article_data['bodies_found']} div(s) with class 'sqs-html-content'")
    print("\nArticle Text:")
    print(article_data['text'])
    print("\nURL:", article_data['url'])
else:
    print("Failed to extract article content")
#===== testing ====================


Title: Spotlight on Brazilian Iron Ore Shipments to the Far East
Date: July 24, 2025
Found 12 div(s) with class 'sqs-html-content'

Article Text:
WEEKLY DRY MARKET MONITOR Spot Rates | Supply-Demand Trends | Port Congestion Overview Week 30, 2025 | Date: July 23, 2025 This week’s Chart Monitor highlights a continued upward trend in Brazilian iron ore shipments. Since the end of Q1, each month's closing volume has exceeded levels seen in previous years. The first quarter of 2025 closed with approximately 70 million tons shipped, followed by over 80 million tons in Q2. These figures are notably higher than the lows recorded in 2021—around 50 million tons in Q1 and 67 million tons in Q2. Looking ahead, recent news from Vale’s Q2 production report adds to the bullish sentiment. The Brazilian miner reported 83.6 million metric tons of iron ore output in Q2, a 3.7% year-on-year increase. The growth was largely attributed to a new second-quarter production record at the S11D mine in northern 

In [17]:
# Initialize the DataFrame
breakwave_article = pd.DataFrame(columns=['Date','Link' , 'Title', 'Content'])

# Loop through each article link
for i, link in enumerate(tqdm.tqdm(breakwave_links_article)):
    # Extract article data for the current link
    article_data = extract_article_text(link)
    
    if article_data:
        # Create a new row as a dictionary
        new_row = {
            'Date': article_data['date'],
            'Link': link,
            'Title': article_data['title'],
            'Content': article_data['text']
        }
        
        # Append the row to the DataFrame
        breakwave_article = pd.concat([breakwave_article, pd.DataFrame([new_row])], ignore_index=True)
    else:
        print(f"Failed to extract article from: {link}")

100%|██████████| 306/306 [05:43<00:00,  1.12s/it]


In [18]:
breakwave_article['Datetime'] = pd.to_datetime(breakwave_article['Date'], errors='coerce')
breakwave_article

,Date,Link,Title,Content,Datetime
0,"February 13, 2026",https://www.breakwaveadvisors.com/insights/202...,Breakwave Weekend Read,Welcome to Breakwave Advisors Weekend Read As ...,2026-02-13
1,"February 6, 2026",https://www.breakwaveadvisors.com/insights/202...,Breakwave Weekend Read,Welcome to Breakwave Advisors Weekend Read As ...,2026-02-06
2,"January 30, 2026",https://www.breakwaveadvisors.com/insights/202...,Breakwave Weekend Read,Welcome to Breakwave Advisors Weekend Read As ...,2026-01-30
3,"January 23, 2026",https://www.breakwaveadvisors.com/insights/202...,Breakwave Weekend Read,Welcome to Breakwave Advisors Weekend Read As ...,2026-01-23
4,"December 23, 2025",https://www.breakwaveadvisors.com/insights/202...,No title found,"In recent years, the dry bulk market has exper...",2025-12-23
...,...,...,...,...,...
301,"June 6, 2025",https://www.breakwaveadvisors.com/insights/202...,Growth Again in New Home Sales in China,By Jeffrey Landsberg As we discussed for clien...,2025-06-06
302,"June 9, 2025",https://www.breakwaveadvisors.com/insights/202...,Capesize Iron Ore Freight Market Trends,"Dry Weekly Market Monitor - Week 23, 2025 Snap...",2025-06-09
303,"June 3, 2025",https://www.breakwaveadvisors.com/insights/202...,No title found,• VLCC Market Under Pressure Amid Summer Lull ...,2025-06-03
304,"June 6, 2025",https://www.breakwaveadvisors.com/insights/202...,Dwindling inventories pushes copper higher,Easing trade tensions helped push the energy s...,2025-06-06


In [ ]:
breakwave_articles_new = pd.read_csv("exported_csv/extracted_articles_new.csv")
#concactenate the new breakwave articles with the existing ones
breakwave_articles_new = pd.concat([breakwave_articles_new, breakwave_article]).drop_duplicates(subset=['Link']).reset_index(drop=True)
breakwave_articles_new.to_csv("exported_csv/breakwave_articles_new.csv", index=False, encoding='utf-8')


## BalticExchange:

In [ ]:
html_content=""" <div class="article-filter-content"> <article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-10-23T00:00:00.000+01:00">23 Oct 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-43.html"><span class="news-title">Bulk report – Week 43</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-10-16T00:00:00.000+01:00">16 Oct 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-42.html"><span class="news-title">Bulk report – Week 42</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-10-09T00:00:00.000+01:00">09 Oct 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-41.html"><span class="news-title">Bulk report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-10-02T00:00:00.000+01:00">02 Oct 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-40.html"><span class="news-title">Bulk report – Week 40</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-09-25T00:00:00.000+01:00">25 Sept 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-39.html"><span class="news-title">Bulk report – Week 39</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-09-18T00:00:00.000+01:00">18 Sept 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-38.html"><span class="news-title">Bulk report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-09-11T00:00:00.000+01:00">11 Sept 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-37.html"><span class="news-title">Bulk report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-09-04T00:00:00.000+01:00">04 Sept 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-36.html"><span class="news-title">Bulk report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-08-28T00:00:00.000+01:00">28 Aug 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-35.html"><span class="news-title">Bulk report – Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-08-21T00:00:00.000+01:00">21 Aug 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-34.html"><span class="news-title">Bulk report – Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-08-14T00:00:00.000+01:00">14 Aug 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-33.html"><span class="news-title">Bulk report – Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-08-07T00:00:00.000+01:00">07 Aug 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-32.html"><span class="news-title">Bulk report – Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-07-31T00:00:00.000+01:00">31 Jul 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-31.html"><span class="news-title">Bulk report – Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-07-24T00:00:00.000+01:00">24 Jul 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-30.html"><span class="news-title">Bulk report – Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-07-17T00:00:00.000+01:00">17 Jul 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-29.html"><span class="news-title">Bulk report – Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-07-10T00:00:00.000+01:00">10 Jul 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-28.html"><span class="news-title">Bulk report – Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-07-03T00:00:00.000+01:00">03 Jul 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-27.html"><span class="news-title">Bulk report – Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-06-26T00:00:00.000+01:00">26 Jun 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-26.html"><span class="news-title">Bulk report – Week 26</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-06-19T00:00:00.000+01:00">19 Jun 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-25.html"><span class="news-title">Bulk report – Week 25</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-06-12T00:00:00.000+01:00">12 Jun 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-24.html"><span class="news-title">Bulk report – Week 24</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-06-05T00:00:00.000+01:00">05 Jun 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-23.html"><span class="news-title">Bulk report – Week 23</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-05-29T00:00:00.000+01:00">29 May 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-22.html"><span class="news-title">Bulk report – Week 22</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-05-22T00:00:00.000+01:00">22 May 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-21.html"><span class="news-title">Bulk report – Week 21</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-05-15T00:00:00.000+01:00">15 May 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-20.html"><span class="news-title">Bulk report – Week 20</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-05-08T00:00:00.000+01:00">08 May 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-19.html"><span class="news-title">Bulk report – Week 19</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-05-01T00:00:00.000+01:00">01 May 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-18.html"><span class="news-title">Bulk report – Week 18</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-04-24T00:00:00.000+01:00">24 Apr 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-17.html"><span class="news-title">Bulk report – Week 17</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-04-17T00:00:00.000+01:00">17 Apr 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-16.html"><span class="news-title">Bulk report – Week 16</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-04-10T00:00:00.000+01:00">10 Apr 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-15.html"><span class="news-title">Bulk report – Week 15</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-03-27T00:00:00.000Z">27 Mar 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-13.html"><span class="news-title">Bulk report – Week 13</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-03-20T00:00:00.000Z">20 Mar 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-12.html"><span class="news-title">Bulk report – Week 12</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-03-13T00:00:00.000Z">13 Mar 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-11.html"><span class="news-title">Bulk report – Week 11</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-03-06T00:00:00.000Z">06 Mar 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-10.html"><span class="news-title">Bulk report – Week 10</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-02-27T00:00:00.000Z">27 Feb 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-9.html"><span class="news-title">Bulk report – Week 9</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-02-20T00:00:00.000Z">20 Feb 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-8.html"><span class="news-title">Bulk report – Week 8</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-02-13T00:00:00.000Z">13 Feb 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-7.html"><span class="news-title">Bulk report – Week 7</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-02-06T00:00:00.000Z">06 Feb 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-6.html"><span class="news-title">Bulk report – Week 6</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-01-30T00:00:00.000Z">30 Jan 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-5.html"><span class="news-title">Bulk report – Week 5</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-01-23T00:00:00.000Z">23 Jan 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-4.html"><span class="news-title">Bulk report – Week 4</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-01-16T00:00:00.000Z">16 Jan 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-3.html"><span class="news-title">Bulk report – Week 3</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2026-01-09T00:00:00.000Z">09 Jan 2026</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2026/bulk-report-week-2.html"><span class="news-title">Bulk report – Week 2</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-12-19T00:00:00.000Z">19 Dec 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-51.html"><span class="news-title">Bulk report – Week 51</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-12-12T00:00:00.000Z">12 Dec 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-50.html"><span class="news-title">Bulk report – Week 50</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-12-05T00:00:00.000Z">05 Dec 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-49.html"><span class="news-title">Bulk report – Week 49</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-11-28T00:00:00.000Z">28 Nov 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-48.html"><span class="news-title">Bulk report – Week 48</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-11-21T00:00:00.000Z">21 Nov 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-47.html"><span class="news-title">Bulk report – Week 47</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-11-14T00:00:00.000Z">14 Nov 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-46.html"><span class="news-title">Bulk report – Week 46</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-11-07T00:00:00.000Z">07 Nov 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-45.html"><span class="news-title">Bulk report – Week 45</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-10-31T00:00:00.000Z">31 Oct 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-44.html"><span class="news-title">Bulk report – Week 44</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-10-24T00:00:00.000+01:00">24 Oct 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-43.html"><span class="news-title">Bulk report – Week 43</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-10-17T00:00:00.000+01:00">17 Oct 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-42.html"><span class="news-title">Bulk report – Week 42</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-10-10T00:00:00.000+01:00">10 Oct 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-41.html"><span class="news-title">Bulk report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-10-03T00:00:00.000+01:00">03 Oct 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-40.html"><span class="news-title">Bulk report – Week 40</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-09-26T00:00:00.000+01:00">26 Sept 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-39.html"><span class="news-title">Bulk report – Week 39</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-09-19T00:00:00.000+01:00">19 Sept 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-38.html"><span class="news-title">Bulk report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-09-12T00:00:00.000+01:00">12 Sept 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-37.html"><span class="news-title">Bulk report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-09-05T00:00:00.000+01:00">05 Sept 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-36.html"><span class="news-title">Bulk report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-08-29T00:00:00.000+01:00">29 Aug 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-35.html"><span class="news-title">Bulk report – Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-08-22T00:00:00.000+01:00">22 Aug 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-34.html"><span class="news-title">Bulk report – Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-08-15T00:00:00.000+01:00">15 Aug 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-33.html"><span class="news-title">Bulk report – Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-08-08T00:00:00.000+01:00">08 Aug 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-32.html"><span class="news-title">Bulk report – Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-08-01T00:00:00.000+01:00">01 Aug 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-31.html"><span class="news-title">Bulk report – Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-07-25T00:00:00.000+01:00">25 Jul 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-30.html"><span class="news-title">Bulk report – Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-07-18T00:00:00.000+01:00">18 Jul 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-29.html"><span class="news-title">Bulk report – Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-07-11T00:00:00.000+01:00">11 Jul 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-28.html"><span class="news-title">Bulk report – Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-07-04T00:00:00.000+01:00">04 Jul 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-27.html"><span class="news-title">Bulk report – Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-06-27T00:00:00.000+01:00">27 Jun 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-26.html"><span class="news-title">Bulk report – Week 26</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-06-20T00:00:00.000+01:00">20 Jun 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-25.html"><span class="news-title">Bulk report – Week 25</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-06-13T00:00:00.000+01:00">13 Jun 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-24.html"><span class="news-title">Bulk report – Week 24</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-06-06T00:00:00.000+01:00">06 Jun 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-23.html"><span class="news-title">Bulk report – Week 23</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-05-30T00:00:00.000+01:00">30 May 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-22.html"><span class="news-title">Bulk report – Week 22</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-05-23T00:00:00.000+01:00">23 May 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-21.html"><span class="news-title">Bulk report – Week 21</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-05-16T00:00:00.000+01:00">16 May 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-20.html"><span class="news-title">Bulk report – Week 20</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-05-09T00:00:00.000+01:00">09 May 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-19.html"><span class="news-title">Bulk report – Week 19</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-05-02T00:00:00.000+01:00">02 May 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-18.html"><span class="news-title">Bulk report – Week 18</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-04-25T00:00:00.000+01:00">25 Apr 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-17.html"><span class="news-title">Bulk report – Week 17</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-04-11T00:00:00.000+01:00">11 Apr 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-15.html"><span class="news-title">Bulk report – Week 15</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-04-04T00:00:00.000+01:00">04 Apr 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-14.html"><span class="news-title">Bulk report – Week 14</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-03-28T00:00:00.000Z">28 Mar 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-13.html"><span class="news-title">Bulk report – Week 13</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-03-21T00:00:00.000Z">21 Mar 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-12.html"><span class="news-title">Bulk report – Week 12</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-03-14T00:00:00.000Z">14 Mar 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-11.html"><span class="news-title">Bulk report – Week 11</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-03-07T00:00:00.000Z">07 Mar 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-10.html"><span class="news-title">Bulk report – Week 10</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-02-28T00:00:00.000Z">28 Feb 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-9.html"><span class="news-title">Bulk report – Week 9</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-02-21T00:00:00.000Z">21 Feb 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-8.html"><span class="news-title">Bulk report – Week 8</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-02-14T00:00:00.000Z">14 Feb 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-7.html"><span class="news-title">Bulk report – Week 7</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-02-07T00:00:00.000Z">07 Feb 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-6.html"><span class="news-title">Bulk report – Week 6</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-01-31T00:00:00.000Z">31 Jan 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-5.html"><span class="news-title">Bulk report – Week 5</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-01-24T00:00:00.000Z">24 Jan 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-4.html"><span class="news-title">Bulk report – Week 4</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-01-17T00:00:00.000Z">17 Jan 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-3.html"><span class="news-title">Bulk report – Week 3</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2025-01-10T00:00:00.000Z">10 Jan 2025</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-2.html"><span class="news-title">Bulk report – Week 2</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-12-20T00:00:00.000Z">20 Dec 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-51.html"><span class="news-title">Bulk report – Week 51</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-12-13T00:00:00.000Z">13 Dec 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-50.html"><span class="news-title">Bulk report – Week 50</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-12-06T00:00:00.000Z">06 Dec 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-49.html"><span class="news-title">Bulk report – Week 49</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-11-29T00:00:00.000Z">29 Nov 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-48.html"><span class="news-title">Bulk report – Week 48</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-11-22T00:00:00.000Z">22 Nov 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-47.html"><span class="news-title">Bulk report – Week 47</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-11-15T00:00:00.000Z">15 Nov 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-46.html"><span class="news-title">Bulk report – Week 46</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-11-08T00:00:00.000Z">08 Nov 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-45.html"><span class="news-title">Bulk report – Week 45</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-11-01T00:00:00.000Z">01 Nov 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-44.html"><span class="news-title">Bulk report – Week 44</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-10-25T00:00:00.000+01:00">25 Oct 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-43.html"><span class="news-title">Bulk report – Week 43</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-10-18T00:00:00.000+01:00">18 Oct 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-42.html"><span class="news-title">Bulk report – Week 42</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-10-11T00:00:00.000+01:00">11 Oct 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-41.html"><span class="news-title">Bulk report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-10-04T00:00:00.000+01:00">04 Oct 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-40.html"><span class="news-title">Bulk report – Week 40</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-09-27T00:00:00.000+01:00">27 Sept 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-39.html"><span class="news-title">Bulk report – Week 39</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-09-20T00:00:00.000+01:00">20 Sept 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-38.html"><span class="news-title">Bulk report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-09-13T00:00:00.000+01:00">13 Sept 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-37.html"><span class="news-title">Bulk report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-09-06T00:00:00.000+01:00">06 Sept 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-36.html"><span class="news-title">Bulk report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-08-30T00:00:00.000+01:00">30 Aug 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-35.html"><span class="news-title">Bulk report – Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-08-23T00:00:00.000+01:00">23 Aug 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-34.html"><span class="news-title">Bulk report – Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-08-16T00:00:00.000+01:00">16 Aug 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-33.html"><span class="news-title">Bulk report – Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-08-09T00:00:00.000+01:00">09 Aug 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-32.html"><span class="news-title">Bulk report – Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-08-02T00:00:00.000+01:00">02 Aug 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-31.html"><span class="news-title">Bulk report – Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-07-26T00:00:00.000+01:00">26 Jul 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-30.html"><span class="news-title">Bulk report – Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-07-19T00:00:00.000+01:00">19 Jul 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-29.html"><span class="news-title">Bulk report – Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-07-12T00:00:00.000+01:00">12 Jul 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-28.html"><span class="news-title">Bulk report – Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-07-05T00:00:00.000+01:00">05 Jul 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-27.html"><span class="news-title">Bulk report – Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-06-28T00:00:00.000+01:00">28 Jun 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-26.html"><span class="news-title">Bulk report – Week 26</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-06-21T00:00:00.000+01:00">21 Jun 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-25.html"><span class="news-title">Bulk report – Week 25</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-06-14T00:00:00.000+01:00">14 Jun 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-24.html"><span class="news-title">Bulk report – Week 24</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-06-07T00:00:00.000+01:00">07 Jun 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-23.html"><span class="news-title">Bulk report – Week 23</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-05-31T00:00:00.000+01:00">31 May 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-22.html"><span class="news-title">Bulk report – Week 22</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-05-24T00:00:00.000+01:00">24 May 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-21.html"><span class="news-title">Bulk report – Week 21</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-05-17T00:00:00.000+01:00">17 May 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-20.html"><span class="news-title">Bulk report – Week 20</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-05-10T00:00:00.000+01:00">10 May 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-19.html"><span class="news-title">Bulk report – Week 19</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-05-03T00:00:00.000+01:00">03 May 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-18.html"><span class="news-title">Bulk report – Week 18</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-04-26T00:00:00.000+01:00">26 Apr 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-17.html"><span class="news-title">Bulk report – Week 17</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-04-19T00:00:00.000+01:00">19 Apr 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-16.html"><span class="news-title">Bulk report – Week 16</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-04-12T00:00:00.000+01:00">12 Apr 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-15.html"><span class="news-title">Bulk report – Week 15</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-04-05T00:00:00.000+01:00">05 Apr 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-14.html"><span class="news-title">Bulk report – Week 14</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-03-22T00:00:00.000Z">22 Mar 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-12.html"><span class="news-title">Bulk report – Week 12</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-03-15T00:00:00.000Z">15 Mar 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-11.html"><span class="news-title">Bulk report – Week 11</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-03-08T00:00:00.000Z">08 Mar 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-10.html"><span class="news-title">Bulk report – Week 10</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-03-01T00:00:00.000Z">01 Mar 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-9.html"><span class="news-title">Bulk report – Week 9</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-02-23T00:00:00.000Z">23 Feb 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-8.html"><span class="news-title">Bulk report – Week 8</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-02-16T00:00:00.000Z">16 Feb 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-7.html"><span class="news-title">Bulk report – Week 7</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-02-09T00:00:00.000Z">09 Feb 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-6.html"><span class="news-title">Bulk report – Week 6</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-02-02T00:00:00.000Z">02 Feb 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-5.html"><span class="news-title">Bulk report – Week 5</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-01-26T00:00:00.000Z">26 Jan 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-4.html"><span class="news-title">Bulk report – Week 4</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-01-19T00:00:00.000Z">19 Jan 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-3.html"><span class="news-title">Bulk report – Week 3</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-01-12T00:00:00.000Z">12 Jan 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-2.html"><span class="news-title">Bulk report – Week 2</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2024-01-05T00:00:00.000Z">05 Jan 2024</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2024/bulk-report-week-1.html"><span class="news-title">Bulk report – Week 1</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-12-15T00:00:00.000Z">15 Dec 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-50.html"><span class="news-title">Bulk report – Week 50</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-12-08T00:00:00.000Z">08 Dec 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-49.html"><span class="news-title">Bulk report – Week 49</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-12-01T00:00:00.000Z">01 Dec 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-48.html"><span class="news-title">Bulk report – Week 48</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-11-24T00:00:00.000Z">24 Nov 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-47.html"><span class="news-title">Bulk report – Week 47</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-11-17T00:00:00.000Z">17 Nov 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-46.html"><span class="news-title">Bulk report – Week 46</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-11-10T00:00:00.000Z">10 Nov 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-45.html"><span class="news-title">Bulk report – Week 45</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-11-03T00:00:00.000Z">03 Nov 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-44.html"><span class="news-title">Bulk report – Week 44</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-10-27T00:00:00.000+01:00">27 Oct 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-43.html"><span class="news-title">Bulk report – Week 43</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-10-20T00:00:00.000+01:00">20 Oct 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-42.html"><span class="news-title">Bulk report – Week 42</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-10-13T00:00:00.000+01:00">13 Oct 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-41.html"><span class="news-title">Bulk report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-10-06T00:00:00.000+01:00">06 Oct 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-40.html"><span class="news-title">Bulk report – Week 40</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-09-29T00:00:00.000+01:00">29 Sept 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-39.html"><span class="news-title">Bulk report – Week 39</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-09-22T00:00:00.000+01:00">22 Sept 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-38.html"><span class="news-title">Bulk report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-09-15T00:00:00.000+01:00">15 Sept 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-37.html"><span class="news-title">Bulk report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-09-08T00:00:00.000+01:00">08 Sept 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-36.html"><span class="news-title">Bulk report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-09-01T00:00:00.000+01:00">01 Sept 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-35.html"><span class="news-title">Bulk report – Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-08-25T00:00:00.000+01:00">25 Aug 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-34.html"><span class="news-title">Bulk report – Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-08-18T00:00:00.000+01:00">18 Aug 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-33.html"><span class="news-title">Bulk report – Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-08-11T00:00:00.000+01:00">11 Aug 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-32.html"><span class="news-title">Bulk report – Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-08-04T00:00:00.000+01:00">04 Aug 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-31.html"><span class="news-title">Bulk report – Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-07-28T00:00:00.000+01:00">28 Jul 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-30.html"><span class="news-title">Bulk report – Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-07-21T00:00:00.000+01:00">21 Jul 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-29.html"><span class="news-title">Bulk report – Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-07-14T00:00:00.000+01:00">14 Jul 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-28.html"><span class="news-title">Bulk report – Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-07-07T00:00:00.000+01:00">07 Jul 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-27.html"><span class="news-title">Bulk report – Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-06-30T00:00:00.000+01:00">30 Jun 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-26.html"><span class="news-title">Bulk report – Week 26</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-06-23T00:00:00.000+01:00">23 Jun 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-25.html"><span class="news-title">Bulk report – Week 25</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-06-16T00:00:00.000+01:00">16 Jun 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-24.html"><span class="news-title">Bulk report – Week 24</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-06-09T00:00:00.000+01:00">09 Jun 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-23.html"><span class="news-title">Bulk report – Week 23</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-06-02T00:00:00.000+01:00">02 Jun 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-22.html"><span class="news-title">Bulk report – Week 22</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-05-26T00:00:00.000+01:00">26 May 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-21.html"><span class="news-title">Bulk report – Week 21</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-05-19T00:00:00.000+01:00">19 May 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-20.html"><span class="news-title">Bulk report – Week 20</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-05-12T00:00:00.000+01:00">12 May 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-19.html"><span class="news-title">Bulk report – Week 19</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-05-05T00:00:00.000+01:00">05 May 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-18.html"><span class="news-title">Bulk report – Week 18</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-04-28T00:00:00.000+01:00">28 Apr 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-17.html"><span class="news-title">Bulk report – Week 17</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-04-21T00:00:00.000+01:00">21 Apr 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-16.html"><span class="news-title">Bulk report – Week 16</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-04-14T00:00:00.000+01:00">14 Apr 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-15.html"><span class="news-title">Bulk report – Week 15</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-03-31T00:00:00.000+01:00">31 Mar 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-13.html"><span class="news-title">Bulk report – Week 13</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-03-24T00:00:00.000Z">24 Mar 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-12.html"><span class="news-title">Bulk report – Week 12</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-03-17T00:00:00.000Z">17 Mar 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-11.html"><span class="news-title">Bulk report – Week 11</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-03-10T00:00:00.000Z">10 Mar 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-10.html"><span class="news-title">Bulk report – Week 10</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-03-03T00:00:00.000Z">03 Mar 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-9.html"><span class="news-title">Bulk report – Week 9</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-02-24T00:00:00.000Z">24 Feb 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-8.html"><span class="news-title">Bulk report – Week 8</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-02-17T00:00:00.000Z">17 Feb 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-7.html"><span class="news-title">Bulk report – Week 7</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-02-10T00:00:00.000Z">10 Feb 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-6.html"><span class="news-title">Bulk report – Week 6</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-02-03T00:00:00.000Z">03 Feb 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-5.html"><span class="news-title">Bulk report – Week 5</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-01-27T00:00:00.000Z">27 Jan 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-4.html"><span class="news-title">Bulk report – Week 4</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-01-20T00:00:00.000Z">20 Jan 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-3.html"><span class="news-title">Bulk report – Week 3</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-01-13T00:00:00.000Z">13 Jan 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-2.html"><span class="news-title">Bulk report – Week 2</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2023-01-06T00:00:00.000Z">06 Jan 2023</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2023/bulk-report-week-1.html"><span class="news-title">Bulk report – Week 1</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-12-16T00:00:00.000Z">16 Dec 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-50.html"><span class="news-title">Bulk report – Week 50</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-12-09T00:00:00.000Z">09 Dec 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-49.html"><span class="news-title">Bulk report – Week 49</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-12-02T00:00:00.000Z">02 Dec 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-48.html"><span class="news-title">Bulk report – Week 48</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-11-25T00:00:00.000Z">25 Nov 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-47.html"><span class="news-title">Bulk report – Week 47</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-11-18T00:00:00.000Z">18 Nov 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-46.html"><span class="news-title">Bulk report – Week 46</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-11-11T00:00:00.000Z">11 Nov 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-45.html"><span class="news-title">Bulk report – Week 45</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-11-04T00:00:00.000Z">04 Nov 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-44.html"><span class="news-title">Bulk report – Week 44</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-10-28T00:00:00.000+01:00">28 Oct 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-43.html"><span class="news-title">Bulk report – Week 43</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-10-21T00:00:00.000+01:00">21 Oct 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-42.html"><span class="news-title">Bulk report – Week 42</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-10-14T00:00:00.000+01:00">14 Oct 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-41.html"><span class="news-title">Bulk report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-10-07T00:00:00.000+01:00">07 Oct 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-40.html"><span class="news-title">Bulk report – Week 40</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-09-30T00:00:00.000+01:00">30 Sept 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-39.html"><span class="news-title">Bulk report – Week 39</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-09-23T00:00:00.000+01:00">23 Sept 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-38.html"><span class="news-title">Bulk report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-09-16T00:00:00.000+01:00">16 Sept 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-37.html"><span class="news-title">Bulk report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-09-09T00:00:00.000+01:00">09 Sept 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-36.html"><span class="news-title">Bulk report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-09-02T00:00:00.000+01:00">02 Sept 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-35.html"><span class="news-title">Bulk report – Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-08-26T00:00:00.000+01:00">26 Aug 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-34.html"><span class="news-title">Bulk report – Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-08-19T00:00:00.000+01:00">19 Aug 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-33.html"><span class="news-title">Bulk report – Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-08-12T00:00:00.000+01:00">12 Aug 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-32.html"><span class="news-title">Bulk report – Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-08-05T00:00:00.000+01:00">05 Aug 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-31.html"><span class="news-title">Bulk report – Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-07-29T00:00:00.000+01:00">29 Jul 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-30.html"><span class="news-title">Bulk report – Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-07-22T00:00:00.000+01:00">22 Jul 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-29.html"><span class="news-title">Bulk report – Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-07-15T00:00:00.000+01:00">15 Jul 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-28.html"><span class="news-title">Bulk report – Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-07-08T00:00:00.000+01:00">08 Jul 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-27.html"><span class="news-title">Bulk report – Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-07-01T00:00:00.000+01:00">01 Jul 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-26.html"><span class="news-title">Bulk report – Week 26</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-06-24T00:00:00.000+01:00">24 Jun 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-25.html"><span class="news-title">Bulk report – Week 25</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-06-17T00:00:00.000+01:00">17 Jun 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-24.html"><span class="news-title">Bulk report – Week 24</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-06-10T00:00:00.000+01:00">10 Jun 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-23.html"><span class="news-title">Bulk report – Week 23</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-05-27T00:00:00.000+01:00">27 May 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-21.html"><span class="news-title">Bulk report – Week 21</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-05-20T00:00:00.000+01:00">20 May 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-20.html"><span class="news-title">Bulk report – Week 20</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-05-13T00:00:00.000+01:00">13 May 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-19.html"><span class="news-title">Bulk report – Week 19</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-05-06T00:00:00.000+01:00">06 May 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-18.html"><span class="news-title">Bulk report – Week 18</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-04-29T00:00:00.000+01:00">29 Apr 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-17.html"><span class="news-title">Bulk report – Week 17</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-04-22T00:00:00.000+01:00">22 Apr 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-16.html"><span class="news-title">Bulk report – Week 16</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-04-08T00:00:00.000+01:00">08 Apr 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-14.html"><span class="news-title">Bulk report – Week 14</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-04-01T00:00:00.000+01:00">01 Apr 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-13.html"><span class="news-title">Bulk report – Week 13</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-03-25T00:00:00.000Z">25 Mar 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-12.html"><span class="news-title">Bulk report – Week 12</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-03-18T00:00:00.000Z">18 Mar 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-11.html"><span class="news-title">Bulk report – Week 11</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-03-11T00:00:00.000Z">11 Mar 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-10.html"><span class="news-title">Bulk report – Week 10</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-03-04T00:00:00.000Z">04 Mar 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-9.html"><span class="news-title">Bulk report – Week 9</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-02-25T00:00:00.000Z">25 Feb 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-8.html"><span class="news-title">Bulk report – Week 8</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-02-18T00:00:00.000Z">18 Feb 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-7.html"><span class="news-title">Bulk report – Week 7</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-02-11T00:00:00.000Z">11 Feb 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-6.html"><span class="news-title">Bulk report – Week 6</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-02-04T00:00:00.000Z">04 Feb 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-5.html"><span class="news-title">Bulk report – Week 5</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-01-28T00:00:00.000Z">28 Jan 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-4.html"><span class="news-title">Bulk report – Week 4</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-01-21T00:00:00.000Z">21 Jan 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-3.html"><span class="news-title">Bulk report – Week 3</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-01-14T00:00:00.000Z">14 Jan 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-2.html"><span class="news-title">Bulk report – Week 2</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2022-01-07T00:00:00.000Z">07 Jan 2022</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2022/bulk-report-week-1.html"><span class="news-title">Bulk report – Week 1</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-12-17T00:00:00.000Z">17 Dec 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-50.html"><span class="news-title">Bulk report – Week 50</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-12-10T00:00:00.000Z">10 Dec 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-49.html"><span class="news-title">Bulk report – Week 49</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-12-03T00:00:00.000Z">03 Dec 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-48.html"><span class="news-title">Bulk report – Week 48</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-11-26T00:00:00.000Z">26 Nov 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-47.html"><span class="news-title">Bulk report – Week 47</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-11-19T00:00:00.000Z">19 Nov 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-46.html"><span class="news-title">Bulk report – Week 46</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-11-12T00:00:00.000Z">12 Nov 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-45.html"><span class="news-title">Bulk report – Week 45</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-11-05T00:00:00.000Z">05 Nov 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-44.html"><span class="news-title">Bulk report – Week 44</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-10-29T00:00:00.000+01:00">29 Oct 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-43.html"><span class="news-title">Bulk report – Week 43</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-10-22T00:00:00.000+01:00">22 Oct 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-42.html"><span class="news-title">Bulk report – Week 42</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-10-15T00:00:00.000+01:00">15 Oct 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-41.html"><span class="news-title">Bulk report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-10-08T00:00:00.000+01:00">08 Oct 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-40.html"><span class="news-title">Bulk report – Week 40</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-10-01T00:00:00.000+01:00">01 Oct 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-39.html"><span class="news-title">Bulk report – Week 39</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-09-24T00:00:00.000+01:00">24 Sept 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-38.html"><span class="news-title">Bulk report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-09-17T00:00:00.000+01:00">17 Sept 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-37.html"><span class="news-title">Bulk report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-09-10T00:00:00.000+01:00">10 Sept 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-36.html"><span class="news-title">Bulk report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-09-03T00:00:00.000+01:00">03 Sept 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-35.html"><span class="news-title">Bulk report – Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-08-27T00:00:00.000+01:00">27 Aug 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-34.html"><span class="news-title">Bulk report – Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-08-20T00:00:00.000+01:00">20 Aug 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-33.html"><span class="news-title">Bulk report – Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-08-13T00:00:00.000+01:00">13 Aug 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-32.html"><span class="news-title">Bulk report – Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-08-06T00:00:00.000+01:00">06 Aug 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-31.html"><span class="news-title">Bulk report – Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-07-30T00:00:00.000+01:00">30 Jul 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-30.html"><span class="news-title">Bulk report – Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-07-23T00:00:00.000+01:00">23 Jul 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-29.html"><span class="news-title">Bulk report – Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-07-16T00:00:00.000+01:00">16 Jul 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-28.html"><span class="news-title">Bulk report – Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-07-09T00:00:00.000+01:00">09 Jul 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-27.html"><span class="news-title">Bulk report – Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-07-02T00:00:00.000+01:00">02 Jul 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-26.html"><span class="news-title">Bulk report – Week 26</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-06-25T00:00:00.000+01:00">25 Jun 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-25.html"><span class="news-title">Bulk report – Week 25</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-06-18T00:00:00.000+01:00">18 Jun 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-24.html"><span class="news-title">Bulk report – Week 24</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-06-11T00:00:00.000+01:00">11 Jun 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-23.html"><span class="news-title">Bulk report – Week 23</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-06-04T00:00:00.000+01:00">04 Jun 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-22.html"><span class="news-title">Bulk report – Week 22</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-05-28T00:00:00.000+01:00">28 May 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-21.html"><span class="news-title">Bulk report – Week 21</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-05-21T00:00:00.000+01:00">21 May 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-20.html"><span class="news-title">Bulk report – Week 20</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-05-14T00:00:00.000+01:00">14 May 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-19.html"><span class="news-title">Bulk report – Week 19</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-05-07T00:00:00.000+01:00">07 May 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-18.html"><span class="news-title">Bulk report – Week 18</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-04-30T00:00:00.000+01:00">30 Apr 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-17.html"><span class="news-title">Bulk report – Week 17</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-04-23T00:00:00.000+01:00">23 Apr 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-16.html"><span class="news-title">Bulk report – Week 16</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-04-16T00:00:00.000+01:00">16 Apr 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-15.html"><span class="news-title">Bulk report – Week 15</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-04-09T00:00:00.000+01:00">09 Apr 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-14.html"><span class="news-title">Bulk report – Week 14</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-03-26T00:00:00.000Z">26 Mar 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-12.html"><span class="news-title">Bulk report – Week 12</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-03-19T00:00:00.000Z">19 Mar 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-11.html"><span class="news-title">Bulk report – Week 11</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-03-12T00:00:00.000Z">12 Mar 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-10.html"><span class="news-title">Bulk report – Week 10</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-03-05T00:00:00.000Z">05 Mar 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-9.html"><span class="news-title">Bulk report – Week 9</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-02-26T00:00:00.000Z">26 Feb 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-8.html"><span class="news-title">Bulk report – Week 8</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-02-19T00:00:00.000Z">19 Feb 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-7.html"><span class="news-title">Bulk report – Week 7</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-02-12T00:00:00.000Z">12 Feb 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-6.html"><span class="news-title">Bulk report – Week 6</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-02-05T00:00:00.000Z">05 Feb 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-5.html"><span class="news-title">Bulk report – Week 5</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-01-29T00:00:00.000Z">29 Jan 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-4.html"><span class="news-title">Bulk report – Week 4</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-01-22T00:00:00.000Z">22 Jan 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-3.html"><span class="news-title">Bulk report – Week 3</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-01-15T00:00:00.000Z">15 Jan 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-2.html"><span class="news-title">Bulk report – Week 2</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2021-01-08T00:00:00.000Z">08 Jan 2021</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2021/bulk-report-week-1.html"><span class="news-title">Bulk report – Week 1</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-12-18T00:00:00.000Z">18 Dec 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-51.html"><span class="news-title">Bulk report – Week 51</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-12-11T00:00:00.000Z">11 Dec 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-50.html"><span class="news-title">Bulk report – Week 50</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-12-04T00:00:00.000Z">04 Dec 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-49.html"><span class="news-title">Bulk report – Week 49</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-11-27T00:00:00.000Z">27 Nov 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-48.html"><span class="news-title">Bulk report – Week 48</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-11-20T00:00:00.000Z">20 Nov 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-47.html"><span class="news-title">Bulk report – Week 47</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-11-13T00:00:00.000Z">13 Nov 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-46.html"><span class="news-title">Bulk report – Week 46</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-11-06T00:00:00.000Z">06 Nov 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-45.html"><span class="news-title">Bulk report – Week 45</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-10-30T00:00:00.000Z">30 Oct 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-44.html"><span class="news-title">Bulk report – Week 44</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-10-23T00:00:00.000+01:00">23 Oct 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-43.html"><span class="news-title">Bulk report – Week 43</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-10-16T00:00:00.000+01:00">16 Oct 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-42.html"><span class="news-title">Bulk report – Week 42</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-10-09T00:00:00.000+01:00">09 Oct 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-41.html"><span class="news-title">Bulk report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-10-02T00:00:00.000+01:00">02 Oct 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-40.html"><span class="news-title">Bulk report – Week 40</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-09-25T00:00:00.000+01:00">25 Sept 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-39.html"><span class="news-title">Bulk report – Week 39</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-09-18T00:00:00.000+01:00">18 Sept 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-38.html"><span class="news-title">Bulk report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-09-11T00:00:00.000+01:00">11 Sept 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-37.html"><span class="news-title">Bulk report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-09-04T00:00:00.000+01:00">04 Sept 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-36.html"><span class="news-title">Bulk report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-08-28T00:00:00.000+01:00">28 Aug 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-35.html"><span class="news-title">Bulk report – Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-08-21T00:00:00.000+01:00">21 Aug 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-34.html"><span class="news-title">Bulk report – Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-08-14T00:00:00.000+01:00">14 Aug 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-33.html"><span class="news-title">Bulk report – Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-08-07T00:00:00.000+01:00">07 Aug 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-32.html"><span class="news-title">Bulk report – Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-07-31T00:00:00.000+01:00">31 Jul 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-31.html"><span class="news-title">Bulk report – Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-07-24T00:00:00.000+01:00">24 Jul 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-30.html"><span class="news-title">Bulk report – Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-07-17T00:00:00.000+01:00">17 Jul 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week-29.html"><span class="news-title">Bulk report – Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-07-10T00:00:00.000+01:00">10 Jul 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/Bulk-report-Week-28.html"><span class="news-title">Bulk report – Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-07-03T00:00:00.000+01:00">03 Jul 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/Bulk-report-Week-27.html"><span class="news-title">Bulk report – Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-06-26T00:00:00.000+01:00">26 Jun 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/Bulk-report-Week-26.html"><span class="news-title">Bulk report – Week 26</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-06-19T00:00:00.000+01:00">19 Jun 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/Bulk-report-Week-25.html"><span class="news-title">Bulk report – Week 25</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-06-12T00:00:00.000+01:00">12 Jun 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/Bulk-report-Week-24.html"><span class="news-title">Bulk report – Week 24</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-06-05T00:00:00.000+01:00">05 Jun 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/Bulk-report-Week-23.html"><span class="news-title">Bulk report – Week 23</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-05-29T00:00:00.000+01:00">29 May 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/Bulk-report-Week-22.html"><span class="news-title">Bulk report – Week 22</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-05-22T00:00:00.000+01:00">22 May 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/Bulk-report-Week-21.html"><span class="news-title">Bulk report – Week 21</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-05-15T00:00:00.000+01:00">15 May 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/Bulk-report-Week-20.html"><span class="news-title">Bulk report – Week 20</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-05-01T00:00:00.000+01:00">01 May 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-18.html"><span class="news-title">Bulk report – Week 18</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-04-24T00:00:00.000+01:00">24 Apr 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-17.html"><span class="news-title">Bulk report – Week 17</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-04-17T00:00:00.000+01:00">17 Apr 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-16.html"><span class="news-title">Bulk report – Week 16</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-04-03T00:00:00.000+01:00">03 Apr 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-14.html"><span class="news-title">Bulk report – Week 14</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-03-27T00:00:00.000Z">27 Mar 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-13.html"><span class="news-title">Bulk report – Week 13</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-03-20T00:00:00.000Z">20 Mar 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-12.html"><span class="news-title">Bulk report – Week 12</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-03-13T00:00:00.000Z">13 Mar 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-11.html"><span class="news-title">Bulk report – Week 11</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-03-06T00:00:00.000Z">06 Mar 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-10.html"><span class="news-title">Bulk report – Week 10</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-02-28T00:00:00.000Z">28 Feb 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-9.html"><span class="news-title">Bulk report – Week 9</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-02-21T00:00:00.000Z">21 Feb 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-8.html"><span class="news-title">Bulk report – Week 8</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-02-14T00:00:00.000Z">14 Feb 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report---week-7.html"><span class="news-title">Bulk report – Week 7</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-02-07T15:16:16.000Z">07 Feb 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week6.html"><span class="news-title">Bulk report – Week 6</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-01-31T14:50:01.000Z">31 Jan 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week5.html"><span class="news-title">Bulk report – Week 5</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-01-24T14:35:06.000Z">24 Jan 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week4.html"><span class="news-title">Bulk report – Week 4</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-01-17T14:10:50.000Z">17 Jan 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week3.html"><span class="news-title">Bulk report – Week 3</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-01-10T17:30:59.000Z">10 Jan 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week2.html"><span class="news-title">Bulk report – Week 2</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2020-01-03T17:30:01.000Z">03 Jan 2020</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2020/bulk-report-week1.html"><span class="news-title">Bulk report – Week 1</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-12-20T17:30:30.000Z">20 Dec 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week51.html"><span class="news-title">Bulk report – Week 51</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-12-13T17:30:18.000Z">13 Dec 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week50.html"><span class="news-title">Bulk report – Week 50</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-12-06T17:30:50.000Z">06 Dec 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week49.html"><span class="news-title">Bulk report – Week 49</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-11-29T17:30:46.000Z">29 Nov 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week48.html"><span class="news-title">Bulk report – Week 48</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-11-22T17:30:40.000Z">22 Nov 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week47.html"><span class="news-title">Bulk report – Week 47</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-11-15T17:30:02.000Z">15 Nov 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week46.html"><span class="news-title">Bulk report – Week 46</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-11-08T17:30:07.000Z">08 Nov 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week45.html"><span class="news-title">Bulk report – Week 45</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-11-01T17:23:14.000Z">01 Nov 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week44.html"><span class="news-title">Bulk report – Week 44</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-10-25T17:02:40.000+01:00">25 Oct 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week43.html"><span class="news-title">Bulk report – Week 43</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-10-18T15:05:53.000+01:00">18 Oct 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week42.html"><span class="news-title">Bulk report – Week 42</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-10-11T17:30:00.000+01:00">11 Oct 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week41.html"><span class="news-title">Bulk report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-10-04T17:30:46.000+01:00">04 Oct 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week40.html"><span class="news-title">Bulk report – Week 40</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-09-27T17:30:13.000+01:00">27 Sept 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week39.html"><span class="news-title">Bulk report – Week 39</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-09-20T17:30:57.000+01:00">20 Sept 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week38.html"><span class="news-title">Bulk report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-09-13T17:30:08.000+01:00">13 Sept 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week37.html"><span class="news-title">Bulk report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-09-06T17:30:44.000+01:00">06 Sept 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week36.html"><span class="news-title">Bulk report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-08-30T17:30:05.000+01:00">30 Aug 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week35.html"><span class="news-title">Bulk report – Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-08-23T17:30:12.000+01:00">23 Aug 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week34.html"><span class="news-title">Bulk report – Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-08-16T17:30:40.000+01:00">16 Aug 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week33.html"><span class="news-title">Bulk report – Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-08-09T17:30:56.000+01:00">09 Aug 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week32.html"><span class="news-title">Bulk report – Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-08-02T17:30:50.000+01:00">02 Aug 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week31.html"><span class="news-title">Bulk report – Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-07-26T14:47:33.000+01:00">26 Jul 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week30.html"><span class="news-title">Bulk report – Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-07-19T17:00:19.000+01:00">19 Jul 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week29.html"><span class="news-title">Bulk report – Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-07-12T17:00:37.000+01:00">12 Jul 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week28.html"><span class="news-title">Bulk report – Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-07-05T15:15:22.000+01:00">05 Jul 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week27.html"><span class="news-title">Bulk report – Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-06-28T14:19:04.000+01:00">28 Jun 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week26.html"><span class="news-title">Bulk report – Week 26</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-06-21T14:32:09.000+01:00">21 Jun 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week25.html"><span class="news-title">Bulk report – Week 25</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-06-14T15:44:29.000+01:00">14 Jun 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week24.html"><span class="news-title">Bulk report – Week 24</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-06-07T14:33:10.000+01:00">07 Jun 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week23.html"><span class="news-title">Bulk report – Week 23</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-05-31T15:20:10.000+01:00">31 May 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week22.html"><span class="news-title">Bulk report – Week 22</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-05-24T15:19:20.000+01:00">24 May 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week21.html"><span class="news-title">Bulk report – Week 21</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-05-17T15:05:49.000+01:00">17 May 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week20.html"><span class="news-title">Bulk report – Week 20</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-05-10T14:33:12.000+01:00">10 May 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week19.html"><span class="news-title">Bulk report – Week 19</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-05-03T14:37:55.000+01:00">03 May 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week18.html"><span class="news-title">Bulk report – Week 18</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-04-26T15:25:27.000+01:00">26 Apr 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week17.html"><span class="news-title">Bulk report – Week 17</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-04-12T15:21:28.000+01:00">12 Apr 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week15.html"><span class="news-title">Bulk report – Week 15</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-04-05T15:18:08.000+01:00">05 Apr 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week14.html"><span class="news-title">Bulk report – Week 14</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-03-29T14:56:38.000Z">29 Mar 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week13.html"><span class="news-title">Bulk report – Week 13</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-03-22T15:13:43.000Z">22 Mar 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week12.html"><span class="news-title">Bulk report – Week 12</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-03-15T14:55:10.000Z">15 Mar 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week11.html"><span class="news-title">Bulk report – Week 11</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-03-08T14:50:44.000Z">08 Mar 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week10.html"><span class="news-title">Bulk report – Week 10</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-03-01T15:40:14.000Z">01 Mar 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week9.html"><span class="news-title">Bulk report – Week 9</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-02-22T16:01:31.000Z">22 Feb 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week8.html"><span class="news-title">Bulk report – Week 8</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-02-15T14:30:27.000Z">15 Feb 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week7.html"><span class="news-title">Bulk report – Week 7</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-02-08T14:37:40.000Z">08 Feb 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week6.html"><span class="news-title">Bulk report – Week 6</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-02-01T14:24:39.000Z">01 Feb 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week5.html"><span class="news-title">Bulk report – Week 5</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-01-25T14:15:40.000Z">25 Jan 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week4.html"><span class="news-title">Bulk report – Week 4</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-01-18T14:29:21.000Z">18 Jan 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week3.html"><span class="news-title">Bulk report – Week 3</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-01-11T14:38:03.000Z">11 Jan 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week2.html"><span class="news-title">Bulk report – Week 2</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2019-01-04T13:27:05.000Z">04 Jan 2019</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2019/bulk-report-week1.html"><span class="news-title">Bulk report – Week 1</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-12-21T15:01:47.000Z">21 Dec 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week51.html"><span class="news-title">Bulk report – Week 51</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-12-14T13:06:07.000Z">14 Dec 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week50.html"><span class="news-title">Bulk report – Week 50</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-12-07T14:22:55.000Z">07 Dec 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week49.html"><span class="news-title">Bulk report – Week 49</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-11-30T14:23:33.000Z">30 Nov 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week48.html"><span class="news-title">Bulk report – Week 48</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-11-23T15:02:15.000Z">23 Nov 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week47.html"><span class="news-title">Bulk report – Week 47</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-11-16T14:15:43.000Z">16 Nov 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week46.html"><span class="news-title">Bulk report – Week 46</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-11-09T14:13:19.000Z">09 Nov 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week45.html"><span class="news-title">Bulk report – Week 45</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-11-02T15:00:41.000Z">02 Nov 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week44.html"><span class="news-title">Bulk report – Week 44</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-10-26T14:27:33.000+01:00">26 Oct 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week43.html"><span class="news-title">Bulk report – Week 43</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-10-19T14:46:49.000+01:00">19 Oct 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week42.html"><span class="news-title">Bulk report – Week 42</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-10-12T15:18:43.000+01:00">12 Oct 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week41.html"><span class="news-title">Bulk report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-10-05T12:57:05.000+01:00">05 Oct 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week40.html"><span class="news-title">Bulk report – Week 40</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-09-28T12:36:32.000+01:00">28 Sept 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week39.html"><span class="news-title">Bulk report – Week 39</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-09-21T13:41:12.000+01:00">21 Sept 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week38.html"><span class="news-title">Bulk report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-09-14T14:43:25.000+01:00">14 Sept 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week37.html"><span class="news-title">Bulk report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-09-07T15:07:34.000+01:00">07 Sept 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week36.html"><span class="news-title">Bulk report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-08-31T13:46:16.000+01:00">31 Aug 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week35.html"><span class="news-title">Bulk report – Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-08-24T15:16:57.000+01:00">24 Aug 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week34.html"><span class="news-title">Bulk report – Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-08-17T13:26:27.000+01:00">17 Aug 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week33.html"><span class="news-title">Bulk report – Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-08-10T14:42:54.000+01:00">10 Aug 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week32.html"><span class="news-title">Bulk report – Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-08-03T14:24:38.000+01:00">03 Aug 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week31.html"><span class="news-title">Bulk report – Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-07-27T14:39:07.000+01:00">27 Jul 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week30.html"><span class="news-title">Bulk report – Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-07-20T15:42:38.000+01:00">20 Jul 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week29.html"><span class="news-title">Bulk report – Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-07-13T14:33:23.000+01:00">13 Jul 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week28.html"><span class="news-title">Bulk report – Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-07-06T12:52:08.000+01:00">06 Jul 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week27.html"><span class="news-title">Bulk report – Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-06-29T13:15:43.000+01:00">29 Jun 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week26.html"><span class="news-title">Bulk report – Week 26</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-06-22T15:52:57.000+01:00">22 Jun 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week25.html"><span class="news-title">Bulk report – Week 25</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-06-15T15:08:13.000+01:00">15 Jun 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week24.html"><span class="news-title">Bulk report – Week 24</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-06-08T14:10:08.000+01:00">08 Jun 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week23.html"><span class="news-title">Bulk report – Week 23</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-06-01T14:50:15.000+01:00">01 Jun 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week22.html"><span class="news-title">Bulk report – Week 22</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-05-25T12:36:32.000+01:00">25 May 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week21.html"><span class="news-title">Bulk report – Week 21</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-05-18T15:44:27.000+01:00">18 May 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week20.html"><span class="news-title">Bulk report – Week 20</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-05-11T14:48:44.000+01:00">11 May 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week19.html"><span class="news-title">Bulk report – Week 19</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-05-04T14:43:01.000+01:00">04 May 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week18.html"><span class="news-title">Bulk report – Week 18</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-04-27T14:40:15.000+01:00">27 Apr 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week17.html"><span class="news-title">Bulk report – Week 17</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-04-20T14:06:18.000+01:00">20 Apr 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week16.html"><span class="news-title">Bulk report – Week 16</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-04-13T14:23:57.000+01:00">13 Apr 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week15.html"><span class="news-title">Bulk report – Week 15</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-04-06T13:24:44.000+01:00">06 Apr 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week14.html"><span class="news-title">Bulk report – Week 14</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-03-23T14:08:32.000Z">23 Mar 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week12.html"><span class="news-title">Bulk report – Week 12</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-03-16T13:44:28.000Z">16 Mar 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week11.html"><span class="news-title">Bulk report – Week 11</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-03-09T14:49:25.000Z">09 Mar 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week10.html"><span class="news-title">Bulk report – Week 10</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-03-02T13:45:39.000Z">02 Mar 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week9.html"><span class="news-title">Bulk report – Week 9</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-02-23T22:46:10.000Z">23 Feb 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week8.html"><span class="news-title">Bulk report – Week 8</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-02-16T22:43:57.000Z">16 Feb 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week7.html"><span class="news-title">Bulk report – Week 7</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-02-09T22:41:32.000Z">09 Feb 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week6.html"><span class="news-title">Bulk report – Week 6</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-02-02T22:39:04.000Z">02 Feb 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week5.html"><span class="news-title">Bulk report – Week 5</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-01-26T13:47:31.000Z">26 Jan 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week4.html"><span class="news-title">Bulk report – Week 4</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-01-19T14:23:11.000Z">19 Jan 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week3.html"><span class="news-title">Bulk report – Week 3</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-01-12T15:28:22.000Z">12 Jan 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week2.html"><span class="news-title">Bulk report – Week 2</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2018-01-05T14:21:37.000Z">05 Jan 2018</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2018/bulk-report-week1.html"><span class="news-title">Bulk report – Week 1</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-12-15T13:46:45.000Z">15 Dec 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week50.html"><span class="news-title">Bulk report – Week 50</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-12-08T14:22:23.000Z">08 Dec 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week49.html"><span class="news-title">Bulk report – Week 49</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-12-01T15:14:50.000Z">01 Dec 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week48.html"><span class="news-title">Bulk report – Week 48</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-11-24T13:13:26.000Z">24 Nov 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week47.html"><span class="news-title">Bulk report – week 47</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-11-17T14:34:20.000Z">17 Nov 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week46.html"><span class="news-title">Bulk report – Week 46</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-11-10T15:32:13.000Z">10 Nov 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week45.html"><span class="news-title">Bulk report – Week 45</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-11-03T14:15:00.000Z">03 Nov 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week44.html"><span class="news-title">Bulk report – Week 44</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-11-01T23:51:32.000Z">01 Nov 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week41.html"><span class="news-title">Bulk Report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-11-01T23:51:32.000Z">01 Nov 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week411.html"><span class="news-title">Bulk Report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-10-27T13:14:55.000+01:00">27 Oct 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week43.html"><span class="news-title">Bulk report – Week 43</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-10-20T11:03:31.000+01:00">20 Oct 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week42.html"><span class="news-title">Bulk report – Week 42</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-10-13T13:26:50.000+01:00">13 Oct 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week410.html"><span class="news-title">Bulk report – Week 41</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-10-06T12:51:43.000+01:00">06 Oct 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week40.html"><span class="news-title">Bulk report – Week 40</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-09-29T11:22:34.000+01:00">29 Sept 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week39.html"><span class="news-title">Bulk report – Week 39</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-09-22T12:08:35.000+01:00">22 Sept 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week38.html"><span class="news-title">Bulk report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-09-15T13:12:39.000+01:00">15 Sept 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week37.html"><span class="news-title">Bulk report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-09-08T11:30:59.000+01:00">08 Sept 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week36.html"><span class="news-title">Bulk report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-09-01T13:31:37.000+01:00">01 Sept 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week35.html"><span class="news-title">Bulk report – Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-08-25T14:39:03.000+01:00">25 Aug 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week34.html"><span class="news-title">Bulk report – Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-08-18T13:12:36.000+01:00">18 Aug 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week33.html"><span class="news-title">Bulk report – Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-08-11T13:33:31.000+01:00">11 Aug 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week32.html"><span class="news-title">Bulk report – Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-08-04T13:33:52.000+01:00">04 Aug 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week31.html"><span class="news-title">Bulk report – Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-07-28T14:18:33.000+01:00">28 Jul 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week30.html"><span class="news-title">Bulk report – Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-07-21T14:52:42.000+01:00">21 Jul 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week29.html"><span class="news-title">Bulk report – Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-07-14T15:07:55.000+01:00">14 Jul 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week28.html"><span class="news-title">Bulk report – Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-07-07T11:25:35.000+01:00">07 Jul 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week27.html"><span class="news-title">Bulk report – Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-06-30T11:03:29.000+01:00">30 Jun 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week26.html"><span class="news-title">Bulk report – Week 26</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-06-23T11:06:40.000+01:00">23 Jun 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week25.html"><span class="news-title">Bulk report – Week 25</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-06-16T11:18:42.000+01:00">16 Jun 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week24.html"><span class="news-title">Bulk report – Week 24</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-06-09T11:26:42.000+01:00">09 Jun 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week23.html"><span class="news-title">Bulk report – Week 23</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-06-02T11:48:14.000+01:00">02 Jun 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week22.html"><span class="news-title">Bulk report – Week 22</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-05-26T12:30:04.000+01:00">26 May 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week21.html"><span class="news-title">Bulk report – Week 21</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-05-19T11:17:54.000+01:00">19 May 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week20.html"><span class="news-title">Bulk report – Week 20</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-05-12T11:14:37.000+01:00">12 May 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week19.html"><span class="news-title">Bulk Report – Week 19</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-05-05T11:32:22.000+01:00">05 May 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week18.html"><span class="news-title">Bulk report – Week 18</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-04-28T11:23:42.000+01:00">28 Apr 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week17.html"><span class="news-title">Bulk report – Week 17</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-04-21T14:19:29.000+01:00">21 Apr 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week16.html"><span class="news-title">Bulk Report – Week 16</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-04-07T11:24:00.000+01:00">07 Apr 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week14.html"><span class="news-title">Bulk report – Week 14</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-03-31T15:30:18.000+01:00">31 Mar 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week13.html"><span class="news-title">Bulk report – Week 13</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-03-24T13:10:58.000Z">24 Mar 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week12.html"><span class="news-title">Bulk Report – Week 12</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-03-17T12:18:22.000Z">17 Mar 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week11.html"><span class="news-title">Bulk report – week 11</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-03-10T16:14:52.000Z">10 Mar 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report--week10.html"><span class="news-title">Bulk report- week 10</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-03-03T15:05:37.000Z">03 Mar 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report--week9.html"><span class="news-title">Bulk report- Week 9</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-02-24T17:09:17.000Z">24 Feb 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week8.html"><span class="news-title">Bulk report – Week 8</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-02-17T15:09:22.000Z">17 Feb 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report--week7.html"><span class="news-title">Bulk report- week 7</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-02-10T14:56:03.000Z">10 Feb 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week6.html"><span class="news-title">Bulk report – Week 6</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-02-03T15:40:24.000Z">03 Feb 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week5.html"><span class="news-title">Bulk report – Week 5</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-01-27T14:33:25.000Z">27 Jan 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week4.html"><span class="news-title">Bulk report – Week 4</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-01-20T12:49:18.000Z">20 Jan 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week3.html"><span class="news-title">Bulk report – Week 3</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-01-13T15:54:36.000Z">13 Jan 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week2.html"><span class="news-title">Bulk report – Week 2</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2017-01-06T16:03:52.000Z">06 Jan 2017</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2017/bulk-report-week1.html"><span class="news-title">Bulk report – Week 1</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-12-16T12:42:02.000Z">16 Dec 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week50.html"><span class="news-title">Bulk report – Week 50</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-12-09T12:58:55.000Z">09 Dec 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week49.html"><span class="news-title">Bulk report – Week 49</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-12-02T16:06:40.000Z">02 Dec 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week48.html"><span class="news-title">Bulk report – Week 48</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-11-25T13:22:54.000Z">25 Nov 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week47.html"><span class="news-title">Bulk report – Week 47</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-11-18T12:01:24.000Z">18 Nov 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week46.html"><span class="news-title">Bulk report – Week 46</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-11-11T16:55:41.000Z">11 Nov 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week45.html"><span class="news-title">Bulk report – Week 45</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-11-04T16:48:08.000Z">04 Nov 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week44.html"><span class="news-title">Bulk report – Week 44</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-10-28T14:33:28.000+01:00">28 Oct 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week43.html"><span class="news-title">Bulk report – Week 43</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-10-21T13:20:55.000+01:00">21 Oct 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week42.html"><span class="news-title">Bulk Report – Week 42</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-10-07T14:09:45.000+01:00">07 Oct 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week40.html"><span class="news-title">Bulk report – Week 40</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-09-30T14:26:55.000+01:00">30 Sept 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week39.html"><span class="news-title">Bulk report – Week 39</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-09-23T12:55:03.000+01:00">23 Sept 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week38.html"><span class="news-title">Bulk Report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-09-16T11:25:09.000+01:00">16 Sept 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week37.html"><span class="news-title">Bulk Report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-09-09T14:11:16.000+01:00">09 Sept 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week36.html"><span class="news-title">Bulk Report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-09-02T13:26:48.000+01:00">02 Sept 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week35.html"><span class="news-title">Bulk Report – Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-08-26T15:46:23.000+01:00">26 Aug 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week34.html"><span class="news-title">Bulk Report – Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-08-19T15:49:15.000+01:00">19 Aug 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week33.html"><span class="news-title">Bulk Report – Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-08-12T13:41:22.000+01:00">12 Aug 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week32.html"><span class="news-title">Bulk Report – Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-08-05T13:12:28.000+01:00">05 Aug 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week31.html"><span class="news-title">Bulk Report – Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-07-29T14:03:12.000+01:00">29 Jul 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week30.html"><span class="news-title">Bulk Report – Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-07-22T10:29:17.000+01:00">22 Jul 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week29.html"><span class="news-title">Bulk Report – Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-07-15T14:47:21.000+01:00">15 Jul 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week28.html"><span class="news-title">Bulk Report – Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-07-08T11:54:26.000+01:00">08 Jul 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week27.html"><span class="news-title">Bulk report – Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-07-01T10:13:22.000+01:00">01 Jul 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week262016.html"><span class="news-title">Bulk report – Week 26 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-06-24T13:58:07.000+01:00">24 Jun 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week252016.html"><span class="news-title">Bulk report – Week 25 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-06-17T10:42:58.000+01:00">17 Jun 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week242016.html"><span class="news-title">Bulk report – Week 24 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-06-10T15:33:51.000+01:00">10 Jun 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week232016.html"><span class="news-title">Bulk report – Week 23 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-06-03T14:13:11.000+01:00">03 Jun 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week222016.html"><span class="news-title">Bulk report – Week 22 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-05-27T11:46:23.000+01:00">27 May 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week212016.html"><span class="news-title">Bulk report – Week 21 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-05-20T10:49:42.000+01:00">20 May 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week202016.html"><span class="news-title">Bulk report – Week 20 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-05-13T13:39:56.000+01:00">13 May 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week192016.html"><span class="news-title">Bulk report – Week 19 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-05-06T14:06:51.000+01:00">06 May 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week182016.html"><span class="news-title">Bulk report – Week 18 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-04-29T14:43:35.000+01:00">29 Apr 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week172016.html"><span class="news-title">Bulk report – Week 17 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-04-22T13:34:01.000+01:00">22 Apr 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week162016.html"><span class="news-title">Bulk report – Week 16 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-04-15T13:48:10.000+01:00">15 Apr 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week152016.html"><span class="news-title">Bulk report – Week 15 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-04-08T11:27:50.000+01:00">08 Apr 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week142016.html"><span class="news-title">Bulk report – Week 14 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-04-01T14:33:06.000+01:00">01 Apr 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week132016.html"><span class="news-title">Bulk report – Week 13 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-03-18T15:41:12.000Z">18 Mar 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week112016.html"><span class="news-title">Bulk report – Week 11 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-03-11T13:07:51.000Z">11 Mar 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week102016.html"><span class="news-title">Bulk report – Week 10 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-03-04T15:34:16.000Z">04 Mar 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week92016.html"><span class="news-title">Bulk report – Week 9 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-02-26T14:15:05.000Z">26 Feb 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week82016.html"><span class="news-title">Bulk report – Week 8 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-02-19T12:58:49.000Z">19 Feb 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week72016.html"><span class="news-title">Bulk report – Week 7 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-02-12T12:38:01.000Z">12 Feb 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week62016.html"><span class="news-title">Bulk report – Week 6 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-02-05T11:47:33.000Z">05 Feb 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week52016.html"><span class="news-title">Bulk report – Week 5 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-01-29T15:07:51.000Z">29 Jan 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week42016.html"><span class="news-title">Bulk report – Week 4 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-01-22T14:22:15.000Z">22 Jan 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week32016.html"><span class="news-title">Bulk report – Week 3 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-01-15T17:15:07.000Z">15 Jan 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week22016.html"><span class="news-title">Bulk report – Week 2 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2016-01-08T15:51:39.000Z">08 Jan 2016</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2016/bulk-report-week12016.html"><span class="news-title">Bulk report – Week 1 2016</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-12-18T15:19:35.000Z">18 Dec 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week512015.html"><span class="news-title">Bulk report – Week 51 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-12-11T12:37:43.000Z">11 Dec 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week502015.html"><span class="news-title">Bulk report – Week 50 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-12-04T15:06:05.000Z">04 Dec 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week492015.html"><span class="news-title">Bulk report – Week 49 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-11-27T16:25:09.000Z">27 Nov 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week482015.html"><span class="news-title">Bulk Report – Week 48 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-11-20T14:33:30.000Z">20 Nov 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week472015.html"><span class="news-title">Bulk report – Week 47 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-11-13T12:41:48.000Z">13 Nov 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week462015.html"><span class="news-title">Bulk report – Week 46 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-11-06T11:23:29.000Z">06 Nov 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week452015.html"><span class="news-title">Bulk report – Week 45 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-10-30T10:38:08.000Z">30 Oct 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week442015.html"><span class="news-title">Bulk report – Week 44 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-10-23T11:03:18.000+01:00">23 Oct 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week432015.html"><span class="news-title">Bulk report – Week 43 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-10-16T14:15:58.000+01:00">16 Oct 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week422015.html"><span class="news-title">Bulk report – Week 42 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-10-09T14:25:11.000+01:00">09 Oct 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week412015.html"><span class="news-title">Bulk Report – Week 41 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-10-02T10:22:16.000+01:00">02 Oct 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week402015.html"><span class="news-title">Bulk Report – Week 40 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-09-25T14:46:53.000+01:00">25 Sept 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week392015.html"><span class="news-title">Bulk Report – Week 39 2015</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-09-18T14:50:25.000+01:00">18 Sept 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week38.html"><span class="news-title">Bulk Report – Week 38</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-09-11T11:23:46.000+01:00">11 Sept 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week37.html"><span class="news-title">Bulk Report – Week 37</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-09-04T13:49:48.000+01:00">04 Sept 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/bulk-report-week36.html"><span class="news-title">Bulk Report – Week 36</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-08-28T13:27:44.000+01:00">28 Aug 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-35.html"><span class="news-title">Week 35</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-08-21T14:08:40.000+01:00">21 Aug 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-34.html"><span class="news-title">Week 34</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-08-14T11:37:53.000+01:00">14 Aug 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-33.html"><span class="news-title">Week 33</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-08-07T13:35:11.000+01:00">07 Aug 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-32.html"><span class="news-title">Week 32</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-07-31T14:00:07.000+01:00">31 Jul 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-31.html"><span class="news-title">Week 31</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-07-24T14:15:41.000+01:00">24 Jul 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-30.html"><span class="news-title">Week 30</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-07-17T11:31:07.000+01:00">17 Jul 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-29.html"><span class="news-title">Week 29</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-07-10T14:07:37.000+01:00">10 Jul 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-28.html"><span class="news-title">Week 28</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-07-03T14:40:37.000+01:00">03 Jul 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-27.html"><span class="news-title">Week 27</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-06-26T16:07:16.000+01:00">26 Jun 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-26.html"><span class="news-title">Week 26</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-06-19T13:44:33.000+01:00">19 Jun 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-25.html"><span class="news-title">Week 25</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-06-12T15:56:44.000+01:00">12 Jun 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-24.html"><span class="news-title">Week 24</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-06-05T15:31:56.000+01:00">05 Jun 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-23.html"><span class="news-title">Week 23</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-05-29T14:14:17.000+01:00">29 May 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-22.html"><span class="news-title">Week 22</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-05-22T15:15:15.000+01:00">22 May 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-21.html"><span class="news-title">Week 21</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-05-15T15:16:45.000+01:00">15 May 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-20.html"><span class="news-title">Week 20</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-05-08T13:36:00.000+01:00">08 May 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-19.html"><span class="news-title">Week 19</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-05-01T13:16:47.000+01:00">01 May 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-18.html"><span class="news-title">Week 18</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-04-24T12:46:06.000+01:00">24 Apr 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-17.html"><span class="news-title">Week 17</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-04-17T13:28:28.000+01:00">17 Apr 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-16.html"><span class="news-title">Week 16</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-04-10T12:42:13.000+01:00">10 Apr 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-15.html"><span class="news-title">Week 15</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-03-27T14:38:34.000Z">27 Mar 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-13.html"><span class="news-title">Week 13</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-03-20T14:43:52.000Z">20 Mar 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-12.html"><span class="news-title">Week 12</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-03-13T13:21:50.000Z">13 Mar 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-11.html"><span class="news-title">Week 11</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-03-06T16:10:58.000Z">06 Mar 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-10.html"><span class="news-title">Week 10</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-02-27T15:24:02.000Z">27 Feb 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-9.html"><span class="news-title">Week 9</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-02-20T13:52:38.000Z">20 Feb 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-8.html"><span class="news-title">Week 8</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-02-13T16:11:27.000Z">13 Feb 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-7.html"><span class="news-title">Week 7</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-02-06T16:39:20.000Z">06 Feb 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/week-6.html"><span class="news-title">Week 6</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-01-30T15:52:04.000Z">30 Jan 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/market-endures-panamaxshockduringdifficultweek.html"><span class="news-title">Market endures panamax shock during difficult week</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-01-23T15:33:00.000Z">23 Jan 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/slight-peaks-andtroughsforcapesizes.html"><span class="news-title">Slight peaks and troughs for capesizes</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-01-16T12:21:27.000Z">16 Jan 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/lengthy-tonnage-listsabound.html"><span class="news-title">Lengthy tonnage lists abound</span></a></div></article><article class="article-item category-news"><div class="event-details-wrapper"><div class="event-date"><span class="icon icon-date"></span>Date:<time datetime="2015-01-09T15:37:39.000Z">09 Jan 2015</time></div></div><div class="news-content-wrapper"><a href="http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2015/outlook-bleak.html"><span class="news-title">Outlook bleak</span></a></div></article></div> """

In [ ]:
def extract_article_data_comprehensive(html_content):
    """
    Extract links, dates, and titles from the HTML content with better error handling
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Find all article items
    articles = soup.find_all('article', class_='article-item')
    
    data = []
    
    for article in articles:
        try:
            # Extract date - look for time element
            time_element = article.find('time')
            if time_element:
                date = time_element.get_text(strip=True)
                # Clean up date (remove "Date:" if present)
                date = re.sub(r'^Date:\s*', '', date)
            else:
                date = "No date found"
            
            # Extract title and link
            title_element = article.find('span', class_='news-title')
            title = title_element.get_text(strip=True) if title_element else "No title found"
            
            link_element = article.find('a')
            link = link_element['href'] if link_element and link_element.has_attr('href') else "No link found"
            
            # Also extract the datetime attribute for more precise date if needed
            datetime_attr = time_element['datetime'] if time_element and time_element.has_attr('datetime') else ""
            
            data.append({
                'Date': date,
                'Datetime': datetime_attr,
                'Title': title,
                'Link': link
            })
            
        except Exception as e:
            print(f"Error processing article: {e}")
            continue
    
    return data


In [ ]:

# Extract the data
import os


article_data = extract_article_data_comprehensive(html_content)

# Create DataFrame
Baltic_exchng_links = pd.DataFrame(article_data)

# Display results
print(f"Extracted {len(Baltic_exchng_links)} articles")
print("\nFirst 10 entries:")
print(Baltic_exchng_links.head(10))

# Get some statistics
print(f"\nDate range: {Baltic_exchng_links['Date'].min()} to {Baltic_exchng_links['Date'].max()}")
print(f"Unique titles: {Baltic_exchng_links['Title'].nunique()}")

# update Baltic_exchng_links.csv with the new data if exists
if os.path.exists('exported_csv/Baltic_exchng_links.csv'):
    existing_data = pd.read_csv('exported_csv/Baltic_exchng_links.csv')
    combined_data = pd.concat([existing_data, Baltic_exchng_links]).drop_duplicates().reset_index(drop=True)
    combined_data.to_csv('exported_csv/Baltic_exchng_links.csv', index=False, encoding='utf-8')
    print("\nData updated in 'exported_csv/Baltic_exchng_links.csv'")
else:
    Baltic_exchng_links.to_csv('exported_csv/Baltic_exchng_links.csv', index=False, encoding='utf-8')
    print("\nData saved to 'exported_csv/Baltic_exchng_links.csv'")

Extracted 583 articles

First 10 entries:
           Date                       Datetime                  Title  \
0   23 Oct 2026  2026-10-23T00:00:00.000+01:00  Bulk report – Week 43   
1   16 Oct 2026  2026-10-16T00:00:00.000+01:00  Bulk report – Week 42   
2   09 Oct 2026  2026-10-09T00:00:00.000+01:00  Bulk report – Week 41   
3   02 Oct 2026  2026-10-02T00:00:00.000+01:00  Bulk report – Week 40   
4  25 Sept 2026  2026-09-25T00:00:00.000+01:00  Bulk report – Week 39   
5  18 Sept 2026  2026-09-18T00:00:00.000+01:00  Bulk report – Week 38   
6  11 Sept 2026  2026-09-11T00:00:00.000+01:00  Bulk report – Week 37   
7  04 Sept 2026  2026-09-04T00:00:00.000+01:00  Bulk report – Week 36   
8   28 Aug 2026  2026-08-28T00:00:00.000+01:00  Bulk report – Week 35   
9   21 Aug 2026  2026-08-21T00:00:00.000+01:00  Bulk report – Week 34   

                                                Link  
0  http://www.balticexchange.com/en/data-services...  
1  http://www.balticexchange.com/en/data-ser

In [ ]:
# Load the CSV file
Baltic_exchng_links = pd.read_csv('exported_csv/Baltic_exchng_links.csv')


In [ ]:
def extract_article_content_robust(url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        }
        
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Multiple strategies for finding title
        title = None
        title_selectors = [
            'h1',
            '.article-title',
            '.title',
            '[class*="title"]',
            'head title'
        ]
        
        for selector in title_selectors:
            element = soup.select_one(selector)
            if element and element.get_text(strip=True):
                title = element.get_text(strip=True)
                break
        
        if not title:
            title = "Title not found"
        
        # Multiple strategies for finding content
        content = None
        content_selectors = [
            '.article-body',
            '.article-content',
            '.content',
            '.post-content',
            '[class*="body"]',
            'article'
        ]
        
        for selector in content_selectors:
            element = soup.select_one(selector)
            if element:
                # Clean up the content
                for unwanted in element(['script', 'style', 'nav', 'footer', 'header', 'aside']):
                    unwanted.decompose()
                
                # Get text content
                text = element.get_text(separator='\n', strip=True)
                if len(text) > 200:  # Reasonable content length
                    content = text
                    break
        
        # If still not found, try to find main content area
        if not content:
            main_selectors = ['main', '#main', '.main']
            for selector in main_selectors:
                element = soup.select_one(selector)
                if element:
                    text = element.get_text(separator='\n', strip=True)
                    if len(text) > 200:
                        content = text
                        break
        
        # Final fallback: get all paragraphs with substantial text
        if not content:
            paragraphs = soup.find_all('p')
            content_text = []
            for p in paragraphs:
                text = p.get_text(strip=True)
                if len(text) > 50:  # Only substantial paragraphs
                    content_text.append(text)
            
            if content_text:
                content = '\n\n'.join(content_text)
            else:
                content = "Content not found"
        
        # Clean up the content
        content = re.sub(r'\s+', ' ', content)
        content = re.sub(r'\n\s*\n', '\n\n', content)
        
        return title, content
        
    except Exception as e:
        return f"Error: {str(e)}", f"Error: {str(e)}"

# Test with a single URL first
test_url = "http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-35.html"
print("Testing with:", test_url)
title, content = extract_article_content_robust(test_url)
print("Title:", title)
print("Content preview:", content)

Testing with: http://www.balticexchange.com/en/data-services/WeeklyRoundup/dry/news/2025/bulk-report-week-35.html
Title: Bulk report – Week 35
Content preview: Capesize The Capesize market delivered a mixed yet broadly steady performance this week, with underlying demand offering support despite some midweek pressure on rates. The Pacific started strongly, driven by strong miner activity and firmer C5 fixtures that helped absorb tonnage. However, momentum faded midweek, with C5 values easing despite continued miner and operator presence, with rates slipping back towards the $10.00 level and then edged up on Friday with fixtures at $10.20. In the Atlantic, South Brazil and West Africa to China maintained consistent engagement, with fixtures broadly ranging between the low-to-mid $24 on C3 for the end of September, while fresh October demand began to surface. The North Atlantic also saw more activity as the week developed, with fronthaul and transatlantic enquiry evident, although rates 

In [ ]:
for i in tqdm.tqdm(range(len(Baltic_exchng_links))):
    url = Baltic_exchng_links.at[i, 'Link']
    title, content = extract_article_content_robust(url)
    Baltic_exchng_links.at[i, 'Extracted_Title'] = title
    Baltic_exchng_links.at[i, 'Content'] = content
    

  0%|          | 0/631 [00:00<?, ?it/s]

100%|██████████| 631/631 [08:19<00:00,  1.26it/s]


In [ ]:
# saving Baltic_exchng_links to existing Baltic_exchng_links_with_content.csv or creating a new one if it doesn't exist
if os.path.exists('exported_csv/Baltic_exchng_links_with_content.csv'):
    existing_data_with_content = pd.read_csv('exported_csv/Baltic_exchng_links_with_content.csv')
    combined_data_with_content = pd.concat([existing_data_with_content, Baltic_exchng_links]).drop_duplicates().reset_index(drop=True)
    combined_data_with_content.to_csv('exported_csv/Baltic_exchng_links_with_content.csv', index=False, encoding='utf-8')

else:
    Baltic_exchng_links.to_csv('exported_csv/Baltic_exchng_links_with_content.csv', index=False, encoding='utf-8')
print("\nData updated in 'exported_csv/Baltic_exchng_links_with_content.csv'")


Data updated in 'exported_csv/Baltic_exchng_links_with_content.csv'


## 3. Splash247 (RSS)

In [20]:
# import feedparser
# import requests
# from bs4 import BeautifulSoup

# RSS_URL = "https://splash247.com/feed/"
# feed = feedparser.parse(RSS_URL)
# for entry in feed.entries:
#     print(entry.title)
#     print(entry.link)
#     html = requests.get(entry.link, timeout=20).text
#     soup = BeautifulSoup(html, "html.parser")
#     article = soup.find("article")
#     if article:
#         text = article.get_text(" ", strip=True)
#         print(text[:500])

In [21]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
}

splash_urls = ["https://splash247.com/category/sector/"] + [f"https://splash247.com/category/sector/page/{i}/" for i in range(2, 2372)] # for concatinating into existing dataframe
# splash_urls = ["https://splash247.com/category/sector/"] + [f"https://splash247.com/category/sector/page/{i}/" for i in range(2, 2328)]

for url in tqdm.tqdm(splash_urls):
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    posts = []
    for details in soup.select("div.post-details"):
        # Title + link
        title_tag = (
            details.select_one("h2 a")
            or details.select_one("h3 a")
            or details.select_one("a")
        )
        title = title_tag.get_text(strip=True) if title_tag else None
        link  = title_tag["href"]              if title_tag else None

        # Date (prefer machine-readable datetime attr)
        date_tag = (
            details.select_one("time")
            or details.select_one(".date")
            or details.select_one(".post-date")
        )
        if date_tag:
            date = date_tag.get("datetime") or date_tag.get_text(strip=True)
        else:
            date = None

        # Author
        author_tag = details.select_one("author-name tie-icon")
        author = author_tag.get_text(strip=True) if author_tag else None

        posts.append({"source": "splash", "title": title, "date": date, "author": author, "link": link})

    splash_df_i = pd.DataFrame(posts)
    splash_df_full = pd.concat([splash_df_full, splash_df_i], ignore_index=True) if 'splash_df_full' in locals() else splash_df_i

splash_df_full['date'] = pd.to_datetime(splash_df_full['date'], errors='coerce').dt.strftime('%B %d, %Y')
#sorting by date asc
splash_df_full = splash_df_full.sort_values(by='date', ascending=True)
splash_df_full.reset_index(drop=True, inplace=True)

100%|██████████| 2371/2371 [41:20<00:00,  1.05s/it] 


In [ ]:
# if splash_df_full.csv exists, then load and concat splash_df_full into it and save , if not create splash_df_full.csv and save it
try:
    existing_df = pd.read_csv("exported_csv/splash_df_full.csv", encoding='utf-8')
    combined_df = pd.concat([existing_df, splash_df_full], ignore_index=True)
    combined_df.drop_duplicates(subset=['title', 'date', 'author', 'link'], inplace=True)
    combined_df.to_csv("exported_csv/splash_df_full.csv", index=False, encoding='utf-8')
except FileNotFoundError:
    splash_df_full.to_csv("exported_csv/splash_df_full.csv", index=False, encoding='utf-8')


In [23]:
# # extract text from each article link and add to the DataFrame
# for index, row in tqdm.tqdm(splash_df_full.iterrows()):
#     try:
#         response = requests.get(row['link'], headers=headers, timeout=20)
#         response.raise_for_status()
#         soup = BeautifulSoup(response.text, "html.parser")
#         article = soup.find("article")
#         if article:
#             text = article.get_text(" ", strip=True)
#             splash_df_full.at[index, 'text'] = text
#         else:
#             splash_df_full.at[index, 'text'] = None
#     except Exception as e:
#         print(f"Error fetching {row['link']}: {e}")
#         splash_df_full.at[index, 'text'] = None
        

In [ ]:
splash_df_full = pd.read_csv("exported_csv/splash_df_full.csv", encoding='utf-8')
# convert date to DateTime and sort in asc order and reset index
splash_df_full['date'] = pd.to_datetime(splash_df_full['date'], errors='coerce').dt.strftime('%B %d, %Y')
splash_df_full = splash_df_full.sort_values(by='date', ascending=True)
splash_df_full = splash_df_full.reset_index(drop=True)

In [26]:
# parallelizing using ThreadPoolExecutor to speed up the process
from concurrent.futures import ThreadPoolExecutor, as_completed

max_workers = 20   # tune down to 10 if you hit rate limits
save_every  = 500  # checkpoint to CSV after every N completions

def fetch_one(args):
    index, row = args
    url = row.get('link')
    if not url or pd.isna(url):
        return index, None
    try:
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        entry = (
            soup.find("div", class_="entry-content entry clearfix")
            or soup.select_one("div.entry-content.entry.clearfix")
            or soup.find("article")
            or soup.find("div", class_="post-content")
            or soup.find("div", class_="content")
        )
        return index, entry.get_text(" ", strip=True) if entry else None
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return index, None

results = {}

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {
        executor.submit(fetch_one, (idx, row)): idx
        # for idx, row in splash_df_full.iterrows()
        # # I only need to fetch the articles that don't have text yet, so I filter the DataFrame to only include those rows
        # for idx, row in pd.DataFrame(splash_df_full[splash_df_full['text'].isna()]).iterrows() 
        for idx, row in pd.DataFrame(splash_df_full).iterrows() 
    }

    completed = 0
    for future in tqdm.tqdm(as_completed(futures), total=len(futures), desc="Fetching articles"):
        idx, text = future.result()
        results[idx] = text
        completed += 1

        if completed % save_every == 0 or completed == len(futures):
            for i, t in results.items():
                splash_df_full.at[i, 'text'] = t
            splash_df_full.to_csv("splash_df_full_with_articles.csv", index=False, encoding="utf-8")
            # print(f"  checkpoint: {completed}/{len(futures)} articles fetched and saved.")

Fetching articles:   1%|          | 500/61260 [01:01<6:59:05,  2.42it/s]

  checkpoint: 500/61260 articles fetched and saved.


Fetching articles:   2%|▏         | 1001/61260 [02:08<12:36:34,  1.33it/s]

  checkpoint: 1000/61260 articles fetched and saved.


Fetching articles:   2%|▏         | 1501/61260 [03:21<24:22:46,  1.47s/it]

  checkpoint: 1500/61260 articles fetched and saved.


... (240 outputs left out)

Fetching articles: 100%|█████████▉| 61000/61260 [2:26:22<00:10, 23.93it/s]

  checkpoint: 61000/61260 articles fetched and saved.


Fetching articles: 100%|██████████| 61260/61260 [2:26:26<00:00,  6.97it/s]

  checkpoint: 61260/61260 articles fetched and saved.


In [ ]:
splash_df_full_with_articles = pd.read_csv("exported_csv/splash_df_full_with_articles.csv", encoding='utf-8')
splash_df_full_with_articles
# add splash_df_full into splash_df_full_with_articles whih are not in splash_df_full_with_articles based on title, date, author and link
splash_df_full_with_articles = pd.concat([splash_df_full_with_articles, splash_df_full]).drop_duplicates(subset=['title', 'date', 'author', 'link']).reset_index(drop=True)
splash_df_full_with_articles.to_csv("exported_csv/splash_df_full_with_articles.csv", index=False, encoding='utf-8')

---

## 4. Hellenic Shipping News



In [28]:
# import requests
# from bs4 import BeautifulSoup

# url = "https://www.hellenicshippingnews.com/category/dry-bulk-market/"
# url = "https://www.hellenicshippingnews.com/category/dry-bulk-market/page/4/"
# html = requests.get(url).text
# soup = BeautifulSoup(html, "html.parser")
# for a in soup.select("h2 a"):
#     print(a.text.strip(), a["href"])

In [29]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import tqdm

# # dry bulk links only
# Hellenic_urls = ["https://www.hellenicshippingnews.com/category/dry-bulk-market/"] + \
#                 [f"https://www.hellenicshippingnews.com/category/dry-bulk-market/page/{i}/" for i in range(2, 200)]

# all top shipping and commodities  stories links
Hellenic_urls = ["https://www.hellenicshippingnews.com/tag/top-stories/"] + \
                [f"https://www.hellenicshippingnews.com/tag/top-stories/page/{i}/" for i in range(2, 400)] +\
                ["https://www.hellenicshippingnews.com/category/commodities/commodity-news/"] + \
                [f"https://www.hellenicshippingnews.com/category/commodities/commodity-news/page/{i}/" for i in range(2, 400)]
                


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
}
Hellenic_df_full = pd.DataFrame(columns=["source", "title", "date", "author", "link"])
for url in tqdm.tqdm(Hellenic_urls, desc="Scraping Hellenic URLs"):
    try:
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()
    except Exception as e:
        print(f"Skipping {url}: {e}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    posts = []
    for details in soup.select("div.post-listing"):
        title_tag = (
            details.select_one("h2 a")
            or details.select_one("h3 a")
            or details.select_one("a")
        )
        title = title_tag.get_text(strip=True) if title_tag else None
        link  = title_tag["href"]              if title_tag else None

        date_tag = details.select_one("span.tie-date")
        date = date_tag.get_text(strip=True) if date_tag else None

        author_tag = details.select_one(".author-name")
        author = author_tag.get_text(strip=True) if author_tag else None

        posts.append({"source": "Hellenic", "title": title, "date": date, "author": author, "link": link})

    Hellenic_df_i = pd.DataFrame(posts)
    Hellenic_df_full = pd.concat([Hellenic_df_full, Hellenic_df_i], ignore_index=True) if 'Hellenic_df_full' in locals() else Hellenic_df_i

Scraping Hellenic URLs:   7%|▋         | 55/798 [01:14<24:38,  1.99s/it]

Skipping https://www.hellenicshippingnews.com/tag/top-stories/page/55/: 503 Server Error: Service Unavailable for url: https://www.hellenicshippingnews.com/tag/top-stories/page/55/


Scraping Hellenic URLs:   7%|▋         | 56/798 [01:15<20:03,  1.62s/it]

Skipping https://www.hellenicshippingnews.com/tag/top-stories/page/56/: 503 Server Error: Service Unavailable for url: https://www.hellenicshippingnews.com/tag/top-stories/page/56/


Scraping Hellenic URLs:   7%|▋         | 57/798 [01:15<16:51,  1.36s/it]

Skipping https://www.hellenicshippingnews.com/tag/top-stories/page/57/: 503 Server Error: Service Unavailable for url: https://www.hellenicshippingnews.com/tag/top-stories/page/57/


... (1,284 outputs left out)

Scraping Hellenic URLs: 100%|█████████▉| 797/798 [12:38<00:00,  1.31it/s]

Skipping https://www.hellenicshippingnews.com/category/commodities/commodity-news/page/398/: 503 Server Error: Service Unavailable for url: https://www.hellenicshippingnews.com/category/commodities/commodity-news/page/398/


Scraping Hellenic URLs: 100%|██████████| 798/798 [12:38<00:00,  1.05it/s]

Skipping https://www.hellenicshippingnews.com/category/commodities/commodity-news/page/399/: 503 Server Error: Service Unavailable for url: https://www.hellenicshippingnews.com/category/commodities/commodity-news/page/399/


In [ ]:
# if Hellenic_df_full.csv exists, then load and concat Hellenic_df_full into it and save , if not create Hellenic_df_full.csv and save it
try:
    existing_df = pd.read_csv("exported_csv/Hellenic_df_full.csv", encoding='utf-8')
    combined_df = pd.concat([existing_df, Hellenic_df_full], ignore_index=True)
    combined_df.drop_duplicates(subset=['title', 'date', 'author', 'link'], inplace=True)
    combined_df.to_csv("exported_csv/Hellenic_df_full.csv", index=False, encoding='utf-8')
except FileNotFoundError:
    Hellenic_df_full.to_csv("exported_csv/Hellenic_df_full.csv", index=False, encoding='utf-8')

In [ ]:
Hellenic_df_full = pd.read_csv("exported_csv/Hellenic_df_full.csv", encoding='utf-8')
Hellenic_df_full

,source,title,date,author,link,text
0,Hellenic,Baltic Dry Index climbs to 2501 up 11 points,30/06/2026,NaN,https://www.hellenicshippingnews.com/baltic-dr...,NaN
1,Hellenic,India’s thermal coal imports drop to 4-year lo...,19/06/2026,NaN,https://www.hellenicshippingnews.com/indias-th...,NaN
2,Hellenic,Baltic Dry Index falls to 2729 down 42 points,11/06/2026,NaN,https://www.hellenicshippingnews.com/baltic-dr...,NaN
3,Hellenic,Baltic Dry Index falls to 3124 down 81 points,03/06/2026,NaN,https://www.hellenicshippingnews.com/baltic-dr...,NaN
4,Hellenic,Baltic Dry Index Falls to Over 2-Week Low,22/05/2026,NaN,https://www.hellenicshippingnews.com/baltic-dr...,NaN
...,...,...,...,...,...,...
336,Hellenic,"LME scrap contracts see losses, trading volume...",04/04/2022,NaN,https://www.hellenicshippingnews.com/lme-scrap...,NaN
337,Hellenic,"Windpower firms, EU steel mills strive to narr...",07/02/2022,NaN,https://www.hellenicshippingnews.com/windpower...,NaN
338,Hellenic,‘Commodities to focus on Chinese economic outl...,03/08/2021,NaN,https://www.hellenicshippingnews.com/commoditi...,NaN
339,Hellenic,The Commodities Feed: Copper supply risks grow,31/07/2021,NaN,https://www.hellenicshippingnews.com/the-commo...,NaN


In [32]:
import requests
from bs4 import BeautifulSoup

def extract_post_content(url):
    """Extract text from the 'post-inner' div of a Hellenic Shipping News article."""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')
        post_inner_div = soup.find('div', class_='post-inner')

        if post_inner_div:
            # Extract all text, but remove extra whitespace and newlines
            content = ' '.join(post_inner_div.stripped_strings)
            return content
        else:
            return "Error: 'post-inner' div not found on this page."

    except requests.exceptions.RequestException as e:
        return f"Error fetching the URL: {e}"

# The for loop that applies the extraction to each link
for link in Hellenic_df_full['link'].dropna().unique():
    print(f"--- Extracting from: {link} ---")
    article_content = extract_post_content(link)
    Hellenic_df_full.at[Hellenic_df_full[Hellenic_df_full['link'] == link].index[0], 'text'] = article_content 
    

--- Extracting from: https://www.hellenicshippingnews.com/baltic-dry-index-climbs-to-2501-up-11-points/ ---


C:\Users\irene\AppData\Local\Temp\ipykernel_10748\64957266.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Error fetching the URL: 503 Server Error: Service Unavailable for url: https://www.hellenicshippingnews.com/baltic-dry-index-climbs-to-2501-up-11-points/' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  Hellenic_df_full.at[Hellenic_df_full[Hellenic_df_full['link'] == link].index[0], 'text'] = article_content


--- Extracting from: https://www.hellenicshippingnews.com/indias-thermal-coal-imports-drop-to-4-year-low-on-local-output/ ---
--- Extracting from: https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2729-down-42-points/ ---
--- Extracting from: https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-3124-down-81-points/ ---
--- Extracting from: https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-over-2-week-low/ ---
--- Extracting from: https://www.hellenicshippingnews.com/dry-bulk-market-returns-to-path-of-growth/ ---
--- Extracting from: https://www.hellenicshippingnews.com/dry-bulk-shipping-australian-coal-exports-on-the-rise-but-chinas-share-diminishing/ ---
--- Extracting from: https://www.hellenicshippingnews.com/mitsui-o-s-k-lines-sees-firming-market-conditions-in-dry-bulk/ ---
--- Extracting from: https://www.hellenicshippingnews.com/baltic-dry-index-up-for-13th-day/ ---
--- Extracting from: https://www.hellenicshippingnews.com/baltic-dry-index-clim

In [33]:
Hellenic_df_full

# if text contains "Error fetching the URL"then make Hellenic_df_full['text'] == NA
Hellenic_df_full['text'] = Hellenic_df_full['text'].apply(lambda x: None if isinstance(x, str) and "Error fetching the URL" in x else x)

In [ ]:
# if Hellenic_df_full_with_articles.csv exists, then load and concat Hellenic_df_full into it and save , if not create Hellenic_df_full_with_articles.csv and save it
try:
    existing_df = pd.read_csv("exported_csv/Hellenic_df_full_with_articles.csv", encoding='utf-8')
    combined_df = pd.concat([existing_df, Hellenic_df_full], ignore_index=True)
    combined_df.drop_duplicates(subset=['title', 'date', 'author', 'link'], inplace=True)
    combined_df.to_csv("exported_csv/Hellenic_df_full_with_articles.csv", index=False, encoding='utf-8')
except FileNotFoundError:
    Hellenic_df_full.to_csv("exported_csv/Hellenic_df_full_with_articles.csv", index=False, encoding='utf-8')

In [ ]:
Hellenic_df_full

,source,title,date,author,link,text
0,Hellenic,Baltic Dry Index climbs to 2501 up 11 points,30/06/2026,NaN,https://www.hellenicshippingnews.com/baltic-dr...,None
1,Hellenic,India’s thermal coal imports drop to 4-year lo...,19/06/2026,NaN,https://www.hellenicshippingnews.com/indias-th...,None
2,Hellenic,Baltic Dry Index falls to 2729 down 42 points,11/06/2026,NaN,https://www.hellenicshippingnews.com/baltic-dr...,None
3,Hellenic,Baltic Dry Index falls to 3124 down 81 points,03/06/2026,NaN,https://www.hellenicshippingnews.com/baltic-dr...,None
4,Hellenic,Baltic Dry Index Falls to Over 2-Week Low,22/05/2026,NaN,https://www.hellenicshippingnews.com/baltic-dr...,None
...,...,...,...,...,...,...
336,Hellenic,"LME scrap contracts see losses, trading volume...",04/04/2022,NaN,https://www.hellenicshippingnews.com/lme-scrap...,None
337,Hellenic,"Windpower firms, EU steel mills strive to narr...",07/02/2022,NaN,https://www.hellenicshippingnews.com/windpower...,None
338,Hellenic,‘Commodities to focus on Chinese economic outl...,03/08/2021,NaN,https://www.hellenicshippingnews.com/commoditi...,None
339,Hellenic,The Commodities Feed: Copper supply risks grow,31/07/2021,NaN,https://www.hellenicshippingnews.com/the-commo...,None


: 

---
## Other

### 3. Seatrade Maritime



In [ ]:
seatr_mar_frst = "https://www.seatrade-maritime.com/maritime-transportation/dry-bulk"
seatr_mar_scnd = "https://www.seatrade-maritime.com/maritime-transportation/dry-bulk?page=2"
seatr_mar_last = "https://www.seatrade-maritime.com/maritime-transportation/dry-bulk?page=93" 

div = "TwoColumnLayout-Content"

date = """<span class="ContentCard-Date" data-testid="content-card-date">May 14, 2026</span>"""

---

### 4. IMF Reports

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.imf.org/en/Publications"

html = requests.get(url).text

soup = BeautifulSoup(html, "html.parser")

for a in soup.find_all("a", href=True):
    href = a["href"]

    if href.endswith(".pdf"):
        print(href)

In [ ]:
# pdf = requests.get(pdf_url)

# with open("report.pdf", "wb") as f:
#     f.write(pdf.content)

---

### 5. World Bank

The World Bank provides structured APIs.

No scraping.

In [ ]:
import requests
url = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.MKTP.CD?format=json"
data = requests.get(url).json()

In [ ]:
# Option 2 (Better): Use the IMF API
# If you only need economic indicators rather than reports:

import requests
url = "https://www.imf.org/external/datamapper/api/v1/NGDP_RPCH"
data = requests.get(url).json()
print(data.keys())

---

### 6. EIA

Excellent API.

Returns JSON.

In [ ]:
url = f"https://api.eia.gov/v2/petroleum/pri/spt/data/?api_key={API_KEY}"

---

### 7. USDA

USDA publishes reports in predictable locations.

Download PDFs.

```
requests.get(pdf_url)
```

Extract.

---

### 8. IMO

Many press releases are HTML.

```
requests.get()
BeautifulSoup()
Extract article
```

---

# I would improve your notebook

Since you've already built one crawler for Breakwave Advisors, I **wouldn't create a separate scraper for every site**.

Instead, I'd define a reusable configuration like:

```
Source(
    name="Splash247",
    rss="https://...",
    article_selector="article",
    title_selector="h1",
    date_selector="time"
)
```

Then one generic crawler:

```
for source in sources:
    download_feed(source)
    download_article()
    extract_text()
    clean()
    embedding()
    database()
```